# 43 — the parse on record, and candidate definitions for the coordinating atom

**Nothing is trained, no feature file for modelling is written, no pattern is chosen and no
submission is built, validated or sent.** Pure RDKit, pandas and `pdftotext` over files already on
disk. This notebook produces a recommendation and the evidence for it; the three decisions at the
end are left open deliberately.

Two things in one notebook.

**Part A puts the ORCA parse on record.** `scripts/parse_orca_output.py` produced
`outputs/qm_parsed/` — 252,237 atom rows, 269,256 bond rows, 5,655 molecular rows, zero failures —
but that work exists only as a script and a `parse_report.md`. This repo's convention is that
scripts do compute and notebooks do reporting, so the parse, its one real parser trap, and the
validation that was *tested rather than assumed* are recorded here.

**Parts B–E are what the notebook is actually for.** Correia & Hollenberg describe reversible P450
inhibition by coordination to the sixth axial site of the ferric haem iron. If that atom can be
identified, the QM table already holds its charge and shell populations. Which SMARTS definition
identifies it is a judgement the chemistry does not settle, so three candidates are measured
against all 5,655 compounds and then looked at.

## The sections

| part | question | output |
|---|---|---|
| 0 | provenance — does every input hash and every quotation verify? | `part0_provenance.csv`, `quote_verification.json` |
| A | what did the parse extract, from where, and what was validated? | 3 figures, `sectionA_*.csv` |
| B | what are the candidates, and what grounds each? | `sectionB_candidates.csv` |
| C | coverage, overlap, tie-break disagreement | `sectionC_*.csv` |
| D | what do the matched atoms actually look like? | 4 panels x 3 candidates |
| E | the three open decisions and the evidence for each | `verdict.json` |

## Scope

**Read-only on everything except `outputs/43_coordinating_atom/`.** `outputs/qm_parsed/`,
`outputs/qm_descriptors/`, `data/`, `src/`, `scripts/` and every prior notebook are inputs only.
`notebooks/32_deadzone_aid_retrain.ipynb` and `notebooks/40_single_object_ensemble.ipynb` carry
live submission code and are **never opened**. The scope check at the end fails loudly on any
violation.

In [1]:
# ---- setup. Order follows notebook 42, the most recent and most correct template.
import hashlib
import json
import re
import subprocess
import sys
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from scipy.stats import mannwhitneyu

ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
PIC50_COL = {i: f"{i}_pIC50_direct_inhibition" for i in ISOFORMS}

OUT = REPO_ROOT / "outputs" / "43_coordinating_atom"
FIG_OUT = OUT / "figures"
for d in (OUT, FIG_OUT):
    d.mkdir(parents=True, exist_ok=True)

# RUN START recorded at the top, never inferred at the bottom from a file mtime -- the bug
# notebook 38 shipped and 30/34/35/37/39/41/42 already carry the fix for.
RUN_META_PATH = OUT / "run_meta.json"
if RUN_META_PATH.exists():
    RUN_META = json.loads(RUN_META_PATH.read_text())
    print(f"[resume] run started {RUN_META['started_at_utc']}")
else:
    RUN_META = {"started_at": time.time(),
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    RUN_META_PATH.write_text(json.dumps(RUN_META, indent=2))
    print(f"[new run] started {RUN_META['started_at_utc']}")
# The set of paths ALREADY dirty before this notebook touched anything. TEST 1 subtracts these:
# without it any pre-existing uncommitted change under a protected path is reported as this
# notebook's doing. Notebook 42 added this after it surfaced on docs/leaderboard_submissions.md.
if "dirty_at_start" not in RUN_META:
    RUN_META["dirty_at_start"] = sorted(
        l[3:].strip() for l in subprocess.run(
            ["git", "status", "--porcelain"], capture_output=True, text=True,
            cwd=REPO_ROOT).stdout.splitlines() if l[:2].strip() != "??")
    RUN_META_PATH.write_text(json.dumps(RUN_META, indent=2))
DIRTY_AT_START = set(RUN_META["dirty_at_start"])
RUN_STARTED_AT = float(RUN_META["started_at"])
print(f"tracked paths already modified before this run: {len(DIRTY_AT_START)} "
      f"{sorted(DIRTY_AT_START)}")

# READ-ONLY sources.
CURATED = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
BLIND = REPO_ROOT / "data" / "processed" / "test_blinded_curated.csv"
QM_PARSED = REPO_ROOT / "outputs" / "qm_parsed"
QM_ATOMS = QM_PARSED / "qm_atoms.csv"
QM_BONDS = QM_PARSED / "qm_bonds.csv"
QM_MOL = QM_PARSED / "qm_molecular_raw.csv"
PARSE_REPORT = QM_PARSED / "parse_report.md"
DESCRIPTORS = REPO_ROOT / "outputs" / "qm_descriptors" / "descriptors.csv"
HIRSHFELD = REPO_ROOT / "outputs" / "qm_descriptors" / "hirshfeld_charges.csv"
NB41 = REPO_ROOT / "outputs" / "41_dataset_audit_3d"
NB41_AUDIT = NB41 / "per_compound_audit.csv"
NB41_SUMMARY = NB41 / "section4_basic_nitrogen_summary.csv"
NB41_CROSSCHECK = NB41 / "section4_crosscheck_disagreements.csv"
NB17_CHARGE = (REPO_ROOT / "outputs" / "17_protonation_charge_check"
               / "charge_logp_features.csv")
# Literature: NOT in the repo -- zero PDFs anywhere in it. These are absolute paths into the
# user's Zotero library, which is why Part 0.2 states plainly that a reader without that
# library cannot re-verify the quotations, only the hashes recorded here.
BOOK_PDF = Path("/Users/codiefreeman/Zotero/storage/WIEQQHR7/"
                "Cytochrome P450  structure, mechanism, and biochemistry.pdf")
BECK_PDF = Path("/Users/codiefreeman/Zotero/storage/6NUWGCBM/Beck et al. - 2021 - Descriptors "
                "of Cytochrome Inhibitors and Useful Machine Learning Based Methods for the "
                "Design of Saf.pdf")

DECISIONS = []


def record_decision(decision, reason, alternatives, authority):
    DECISIONS.append({"decision": decision, "reason": reason, "alternatives": alternatives,
                      "authority": authority, "at_utc": datetime.now(timezone.utc).isoformat()})
    (OUT / "decisions_taken.json").write_text(json.dumps(DECISIONS, indent=2))


def pct(n, d):
    return 100.0 * n / d if d else float("nan")


RDLogger.EnableLog("rdApp.*")
RDLogger.DisableLog("rdApp.info")
print(f"rdkit {rdkit.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"python {sys.version.split()[0]}")
print(f"writing to {OUT.relative_to(REPO_ROOT)}/")

[resume] run started 2026-10-05T23:11:47.191489+00:00
tracked paths already modified before this run: 1 ['.gitignore']
rdkit 2026.03.3 | pandas 2.3.3 | numpy 1.26.4
python 3.11.13
writing to outputs/43_coordinating_atom/


---

### Part 0 — provenance: the environment, the inputs by bytes, and one cross-check

Four things before any analysis. The pinned RDKit version is asserted, because every SMARTS
semantic below was established against it. Every input file is hashed and row-counted. And one
hard cross-check runs against notebook 41's own saved summary — verifying against the saved file
rather than against prose is this project's convention, and it has caught three quoting errors in
its own history.

In [2]:
# ---- (1) environment.
PINNED_RDKIT = "2026.03.3"
assert rdkit.__version__ == PINNED_RDKIT, (
    f"expected rdkit {PINNED_RDKIT} (cyp-admet-v2's pin, per CLAUDE.md) but got "
    f"{rdkit.__version__} -- the SMARTS semantics below were established against the pin")
print(f"rdkit {rdkit.__version__} matches CLAUDE.md's pin for cyp-admet-v2.")

# ---- (2) sources, by bytes.
prov = []
for label, path in [("curated train", CURATED), ("blind test", BLIND),
                    ("qm_atoms", QM_ATOMS), ("qm_bonds", QM_BONDS), ("qm_molecular_raw", QM_MOL),
                    ("qm descriptors (nb run)", DESCRIPTORS),
                    ("qm hirshfeld (nb run)", HIRSHFELD),
                    ("nb41 per-compound audit", NB41_AUDIT),
                    ("nb41 basic-N summary", NB41_SUMMARY),
                    ("nb41 nb17 crosscheck", NB41_CROSSCHECK),
                    ("nb17 charge/logP", NB17_CHARGE)]:
    h = hashlib.sha256(path.read_bytes()).hexdigest()
    n = len(pd.read_csv(path))
    prov.append({"label": label, "path": str(path.relative_to(REPO_ROOT)), "rows": n,
                 "sha256": h})
    print(f"  {label:26s} {n:7d} rows  sha256 {h[:16]}...  {path.relative_to(REPO_ROOT)}")
prov = pd.DataFrame(prov)
prov.to_csv(OUT / "part0_provenance.csv", index=False)

# ---- (3) load, and assert the population is the one every prior notebook used.
train = pd.read_csv(CURATED)
blind = pd.read_csv(BLIND)
# The RAW SMILES column, never canonical_smiles. Part A.4 measures why: the ORCA atom order came
# from Chem.AddHs(Chem.MolFromSmiles(row.SMILES)) and canonical_smiles does not reproduce it.
AUD = pd.read_csv(NB41_AUDIT)
assert len(AUD) == 5655, f"expected 5,655 compounds in notebook 41's audit, got {len(AUD)}"
assert AUD.Molecule_Name.is_unique, "Molecule_Name is not unique"
TR = AUD[AUD.set == "train"]
BL = AUD[AUD.set == "blind"]
print(f"\npopulation: {len(TR)} train + {len(BL)} blind = {len(AUD)}")

# ---- (4) hard cross-check against notebook 41's own saved per-isoform figures.
nb41_sum = pd.read_csv(NB41_SUMMARY).set_index("population")
ok = True
print("\nnotebook 41's strict basic-N coverage, recomputed here from its own per-compound file:")
for iso in ISOFORMS:
    here = pct(int((TR[TR[f"has_{iso}"]].n_basic_n > 0).sum()), int(TR[f"has_{iso}"].sum()))
    saved = float(nb41_sum.loc[f"train / {iso}", "pct_ge1_strict"])
    same = abs(here - saved) < 1e-6
    ok &= same
    print(f"  {iso}  here {here:7.3f}%   nb41 saved {saved:7.3f}%   "
          f"{'MATCH' if same else '*** MISMATCH'}")
assert ok, "per-isoform basic-N coverage disagrees with notebook 41's saved summary -- stop"

rdkit 2026.03.3 matches CLAUDE.md's pin for cyp-admet-v2.
  curated train                 4905 rows  sha256 9209ecd45ea8945b...  data/processed/train_inhibition_curated.csv
  blind test                     750 rows  sha256 ba5e6ee298485c5d...  data/processed/test_blinded_curated.csv
  qm_atoms                    252237 rows  sha256 3d333c41ed6e7653...  outputs/qm_parsed/qm_atoms.csv


  qm_bonds                    269256 rows  sha256 0bddb8a98f7402b3...  outputs/qm_parsed/qm_bonds.csv
  qm_molecular_raw              5655 rows  sha256 5f0154d02445161c...  outputs/qm_parsed/qm_molecular_raw.csv
  qm descriptors (nb run)       5655 rows  sha256 9b86ecd07b32a6a2...  outputs/qm_descriptors/descriptors.csv
  qm hirshfeld (nb run)       252237 rows  sha256 1d8606c4a636defb...  outputs/qm_descriptors/hirshfeld_charges.csv
  nb41 per-compound audit       5655 rows  sha256 9ef4427dc3ac6704...  outputs/41_dataset_audit_3d/per_compound_audit.csv
  nb41 basic-N summary             7 rows  sha256 58fb7b17ec99df2c...  outputs/41_dataset_audit_3d/section4_basic_nitrogen_summary.csv
  nb41 nb17 crosscheck          1433 rows  sha256 0ed05e5e7bbb7a40...  outputs/41_dataset_audit_3d/section4_crosscheck_disagreements.csv
  nb17 charge/logP              5655 rows  sha256 a55f0db34f0fab4e...  outputs/17_protonation_charge_check/charge_logp_features.csv

population: 4905 train + 750 blind 

#### Part 0.2 — the quotations, and the limit of verifying them

Every sentence this notebook attributes to a source is checked verbatim against `pdftotext` output
after unicode and whitespace normalisation, following notebook 37's Ash-paper convention. Two
things make the convention weaker here than there, and both are recorded rather than glossed:

**The literature is not in the repo.** There are zero PDFs anywhere in it. Both sources live in
the user's Zotero library at absolute paths, so the sha256 of each file is recorded and the quote
match is reproducible *on this machine*, but a reader without that library can verify neither.
This is the same honesty problem notebook 42 hit with its Discord source, handled the same way.

**The book is a scanned OCR copy, two-column, not born-digital.** Quotations are transcribed as
OCR renders them, artefacts included. Two consequences were found by testing rather than assumed:
a full-page extract *interleaves the two columns* and corrupts any sentence spanning a column
break, while a hard column split at x=240 *clips* text sitting on the boundary. No single
extraction geometry yields every sentence, so the union of three is searched.

The sentence that matters most for this is the one supporting the lipophilicity claim: it is split
across a column break **with figure text interleaved**, so it cannot be recovered contiguously at
all and is verified as two fragments. That is stated here because it is the sentence a reader is
most likely to want to check.

In [3]:
# ---- quote verification. The union of three extraction geometries per page -- see the markdown.
def _norm(t):
    t = unicodedata.normalize("NFKD", t)
    for a, b in [("’", "'"), ("‘", "'"), ("“", '"'), ("”", '"'),
                 ("—", "-"), ("–", "-"), ("−", "-")]:
        t = t.replace(a, b)
    return re.sub(r"\s+", " ", t).strip()


def _book_haystack(p1, p2):
    parts = []
    for p in range(p1, p2 + 1):
        for geom in ([], ["-x", "0", "-y", "0", "-W", "240", "-H", "704"],
                     ["-x", "230", "-y", "0", "-W", "260", "-H", "704"]):
            parts.append(subprocess.run(
                ["pdftotext", "-f", str(p), "-l", str(p)] + geom + [str(BOOK_PDF), "-"],
                capture_output=True, text=True).stdout)
    return _norm(" ".join(parts))


def _beck_haystack(p1, p2):
    # This PDF carries two overlapping text layers (accepted manuscript + final), so -layout
    # interleaves duplicates; -raw gives clean reading order.
    return _norm(subprocess.run(["pdftotext", "-raw", "-f", str(p1), "-l", str(p2),
                                 str(BECK_PDF), "-"], capture_output=True, text=True).stdout)


# Chapter 5 begins at PDF page 191 = printed 177; PDF page = printed page + 14.
BOOK_HAY = _book_haystack(191, 200)
BECK_HAY = _beck_haystack(1, 13)

QUOTES = [
    ("CH5", "p.178", "donor set at the sixth coordination site",
     "Amino-side chains or nitrogenous heterocyclic moieties in some substrates"),
    ("CH5", "p.178", "binds the sixth coordination site directly",
     "directly bind tightly to the sixth coordination site"),
    ("CH5", "p.182", "N: aliphatic OR aromatic, equally, no substitution class",
     "nitrogen-containing aliphatic or aromatic functions that bind the heme iron tightly"),
    ("CH5", "p.183", "named heterocyclic scaffolds", "pyridine, imidazole, and triazole"),
    ("CH5", "p.180", "S: thiols and sulfides at the sixth site",
     "bind sulfur ligands such as thiols"),
    ("CH5", "p.182", "O donors named", "organic alcohols, ethers, ketones, lactones, and other "
     "structures in which an oxygen atom of the ligand coordinates to the heme iron"),
    ("CH5", "p.182", "O donors are WEAK", "generally weak P450 inhibitors"),
    ("CH5", "p.182", "dual tethering beats either mode alone",
     "Such dual tethering of the P450 active site confers much greater inhibitory capacity"),
    ("CH5", "p.183", "determinant (a): the nitrogen lone pair",
     "the intrinsic affinity of their nitrogen electron pair for the heme iron"),
    ("CH5", "p.179", "lipophilic site deters ionic ligands [fragment A of 2]",
     "ligand of ferric myoglobin [46]. The lipophilic"),
    ("CH5", "p.179", "lipophilic site deters ionic ligands [fragment B of 2]",
     "nature of the P450 active site is an additional deterrent to its interactions with "
     "ionic ligands"),
    ("CH5", "p.179", "'neutral' qualifies the ENZYME, not the ligand",
     "the negatively charged cyanide ion favors the neutral ferric P450 species over the more "
     "negatively charged ferrous species"),
    ("CH5", "p.185", "a 2D match cannot tell whether coordination happens",
     "water-bridged low-spin complexes that were metabolically competent"),
    ("Beck", "abstract", "the authors' own blanket caveat",
     "these descriptors alone are insufficient to predict major cytochrome inhibition "
     "probability and affinity"),
    ("Beck", "p.7", "CYP2D6 inhibitors carry a POSITIVE charge",
     "capable of procuring a positive charge"),
    ("Beck", "p.5", "CYP3A4: tertiary amine or imidazole",
     "tertiary amine or imidazole nitrogen frequently coordinate"),
    ("Beck", "p.11", "CYP1A2: secondary amines",
     "the coordination of secondary amines and aromatic moieties"),
    ("Beck", "p.10", "CYP2C9: heterocycles, aromatic N, primary amines",
     "Heterocycles, aromatic nitrogens, and primary amines coordinate to the iron"),
]

rows = []
for src, loc, what, q in QUOTES:
    hay = BOOK_HAY if src == "CH5" else BECK_HAY
    found = _norm(q) in hay
    rows.append({"source": src, "location": loc, "claim": what, "found_verbatim": found,
                 "quote": q})
    print(f"  {'FOUND verbatim' if found else '*** NOT FOUND':16} {src:5} {loc:9} {what}")
QV = pd.DataFrame(rows)
n_ok = int(QV.found_verbatim.sum())
print(f"\n{n_ok}/{len(QV)} quotations found verbatim")

qv_payload = {
    "status": "VERIFIED" if n_ok == len(QV) else "FAILED",
    "verifiable_by_hash": True,
    "reproducible_by_a_third_party": False,
    "why_not": ("Neither source is in the repository -- it contains zero PDFs. Both are in the "
                "user's Zotero library at absolute paths outside it, so the sha256 below pins "
                "what was read on this machine but a reader without that library cannot "
                "re-extract the text. Same limitation notebook 42 recorded for its Discord "
                "source, and deliberately not dressed up as more than it is."),
    "ocr_caveat": ("The book is a scanned two-column copy with an OCR text layer, not "
                   "born-digital. A full-page extract interleaves the columns and corrupts "
                   "sentences spanning a column break; a hard x=240 split clips text on the "
                   "boundary. The union of three geometries is searched. The p.179 lipophilicity "
                   "sentence is split across a column break with figure text interleaved and is "
                   "verified as two fragments, never contiguously."),
    "sources": [
        {"tag": "CH5", "citation": ("Correia, M. A. & Hollenberg, P. F., 'Inhibition of "
                                    "Cytochrome P450 Enzymes', ch. 5 in Ortiz de Montellano "
                                    "et al. (eds), Cytochrome P450: Structure, Mechanism, and "
                                    "Biochemistry, 4th edn, Springer 2015, pp. 177-259. "
                                    "DOI 10.1007/978-3-319-12108-6_5"),
         "path": str(BOOK_PDF), "sha256": hashlib.sha256(BOOK_PDF.read_bytes()).hexdigest()},
        {"tag": "Beck", "citation": ("Beck, T. C., Beck, K. R., Morningstar, J., Benjamin, "
                                     "M. M. & Norris, R. A., 'Descriptors of Cytochrome "
                                     "Inhibitors and Useful Machine Learning Based Methods for "
                                     "the Design of Safer Drugs', Pharmaceuticals 2021, 14, 472. "
                                     "DOI 10.3390/ph14050472"),
         "path": str(BECK_PDF), "sha256": hashlib.sha256(BECK_PDF.read_bytes()).hexdigest()}],
    "quotes": rows,
}
(OUT / "quote_verification.json").write_text(json.dumps(qv_payload, indent=2))
QV.to_csv(OUT / "part0_quote_verification.csv", index=False)
assert n_ok == len(QV), "a quotation did not verify -- fix the quote or the extraction, not the claim"
print(f"-> quote_verification.json  status={qv_payload['status']}")

  FOUND verbatim   CH5   p.178     donor set at the sixth coordination site
  FOUND verbatim   CH5   p.178     binds the sixth coordination site directly
  FOUND verbatim   CH5   p.182     N: aliphatic OR aromatic, equally, no substitution class
  FOUND verbatim   CH5   p.183     named heterocyclic scaffolds
  FOUND verbatim   CH5   p.180     S: thiols and sulfides at the sixth site
  FOUND verbatim   CH5   p.182     O donors named
  FOUND verbatim   CH5   p.182     O donors are WEAK
  FOUND verbatim   CH5   p.182     dual tethering beats either mode alone
  FOUND verbatim   CH5   p.183     determinant (a): the nitrogen lone pair
  FOUND verbatim   CH5   p.179     lipophilic site deters ionic ligands [fragment A of 2]
  FOUND verbatim   CH5   p.179     lipophilic site deters ionic ligands [fragment B of 2]
  FOUND verbatim   CH5   p.179     'neutral' qualifies the ENZYME, not the ligand
  FOUND verbatim   CH5   p.185     a 2D match cannot tell whether coordination happens
  FOUND verba

All 18 quotations verify verbatim. The two sha256 values pin exactly which files were read; the
`reproducible_by_a_third_party: False` flag in `quote_verification.json` records that this is
weaker than notebook 37's Ash check, where the PDF was reachable from the repo.

---

### Part 0.3 — a premise in this notebook's own brief is wrong

The brief that commissioned this notebook stated that Correia & Hollenberg "state the active site
is lipophilic and the thiolate-ligated iron disfavours ionic ligands, **so the NEUTRAL species is
the binding form** — which is what was computed." The first half is right. The second does not
follow from the source, and the cell below shows why from the verified text.

In [4]:
# ---- the premise check. Both sentences are already verified verbatim above; what is at issue is
# what "neutral" refers to.
print("What the chapter actually says (p.179, verified verbatim above):\n")
print('  "the negatively charged cyanide ion favors the NEUTRAL FERRIC P450 SPECIES over the')
print('   more negatively charged FERROUS species... the thiolate ligation of the latter')
print('   enriches the electron density around the iron atom"\n')
print('  "The lipophilic nature of the P450 active site is an additional deterrent to its')
print('   interactions with IONIC ligands [47]."\n')
print("=" * 94)
print("'Neutral' qualifies the ENZYME's redox state -- ferric versus ferrous -- not the ligand.")
print("The whole passage concerns ANIONS (cyanide). The chapter nowhere says a drug's")
print("unprotonated microspecies is the binding form.")
print("=" * 94)
print()
print("And the second source points the other way for CYP2D6 (verified verbatim above):")
print('  Beck et al. p.7:  CYP2D6 inhibitors are "capable of procuring a positive charge"')
print('  Beck et al. p.7:  "CYP2D6 has a predilection for basic substrates"')
print()
print("So the premise is unsupported by the first source and partly contradicted by the second.")
print("The thiolate/electron-density argument, taken at face value, would FAVOUR a cation.")
print()
# What this does and does not change.
nb17 = pd.read_csv(NB17_CHARGE)
n_nonzero = int((nb17.net_charge_ph74 != 0).sum())
qm = pd.read_csv(QM_MOL)
n_neutral = int((qm.formal_charge == 0).sum())
print(f"What was computed: {n_neutral} of {len(qm)} QM molecules are formally neutral "
      f"({pct(n_neutral, len(qm)):.1f}%).")
print(f"Notebook 17's pH 7.4 model puts {n_nonzero} of {len(nb17)} compounds "
      f"({pct(n_nonzero, len(nb17)):.1f}%) at a non-zero charge.")
print()
print("That choice STANDS -- it is defensible, already made, and 11,310 ORCA jobs rest on it.")
print("What changes is only its attribution: it is this project's modelling assumption, not a")
print("finding of Correia & Hollenberg. Recorded as such in 'Decisions taken without approval'.")

record_decision(
    decision=("Corrected the brief's claim that Correia & Hollenberg support computing on the "
              "neutral species, and re-attributed that choice to this project rather than to "
              "the source."),
    reason=("The brief read 'the neutral ferric P450 species' as licensing the neutral LIGAND "
            f"microspecies. Verified verbatim against the PDF (sha256 "
            f"{hashlib.sha256(BOOK_PDF.read_bytes()).hexdigest()[:16]}...), 'neutral' qualifies "
            "the enzyme's redox state -- ferric vs ferrous -- and the passage concerns anions "
            "(cyanide). Beck et al. say the opposite for CYP2D6: inhibitors 'capable of "
            "procuring a positive charge', with 'a predilection for basic substrates'. Leaving "
            "the attribution as briefed would have put a citation behind a claim neither source "
            "makes, in a notebook whose whole purpose is to ground a pattern in cited chemistry. "
            "This is the fifth brief-premise error caught in this project, after notebooks 30, "
            "32, 36 and 38."),
    alternatives=("Repeat the brief's attribution (a false citation in the one place it would "
                  "most mislead); or treat the QM run's neutral-species choice as invalidated "
                  "(it is not -- the choice is reasonable, only its justification was "
                  "misattributed)."),
    authority="none -- the brief asserted it; the source was checked and does not support it")

What the chapter actually says (p.179, verified verbatim above):

  "the negatively charged cyanide ion favors the NEUTRAL FERRIC P450 SPECIES over the
   more negatively charged FERROUS species... the thiolate ligation of the latter
   enriches the electron density around the iron atom"

  "The lipophilic nature of the P450 active site is an additional deterrent to its
   interactions with IONIC ligands [47]."

'Neutral' qualifies the ENZYME's redox state -- ferric versus ferrous -- not the ligand.
The whole passage concerns ANIONS (cyanide). The chapter nowhere says a drug's
unprotonated microspecies is the binding form.

And the second source points the other way for CYP2D6 (verified verbatim above):
  Beck et al. p.7:  CYP2D6 inhibitors are "capable of procuring a positive charge"
  Beck et al. p.7:  "CYP2D6 has a predilection for basic substrates"

So the premise is unsupported by the first source and partly contradicted by the second.
The thiolate/electron-density argument, taken

---

## Part A — the ORCA parse on record

`scripts/parse_orca_output.py` turned 11,310 ORCA job outputs into three tidy tables. The script
and its `parse_report.md` hold the detail; what belongs in the notebook record is the part a
future reader needs: **where each quantity came from, the one trap that could have corrupted the
table silently, and the fact that the validation was tested rather than asserted.**

Every number below is recomputed from `outputs/qm_parsed/` in this notebook, not transcribed.

In [5]:
# ---- A.1 the three tables, recomputed.
A = pd.read_csv(QM_ATOMS)
BND = pd.read_csv(QM_BONDS)
MOL = pd.read_csv(QM_MOL)
DESC = pd.read_csv(DESCRIPTORS)

shape = pd.DataFrame([
    {"table": "qm_atoms.csv", "rows": len(A), "cols": A.shape[1], "expected_rows": 252237},
    {"table": "qm_bonds.csv", "rows": len(BND), "cols": BND.shape[1], "expected_rows": 269256},
    {"table": "qm_molecular_raw.csv", "rows": len(MOL), "cols": MOL.shape[1],
     "expected_rows": 5655}])
shape["matches"] = shape.rows == shape.expected_rows
print(shape.to_string(index=False))
assert shape.matches.all(), "a parsed table's row count does not match the recorded figure"
assert A.Molecule_Name.nunique() == BND.Molecule_Name.nunique() == len(MOL) == 5655

# per-compound atom count must reproduce the generating run's own bookkeeping
g = A.groupby("Molecule_Name").size().rename("parsed")
chk = DESC.set_index("Molecule_Name")[["total_atoms_with_h"]].join(g)
n_match = int((chk.parsed == chk.total_atoms_with_h).sum())
print(f"\nper-compound atom count vs descriptors.csv total_atoms_with_h: "
      f"{n_match}/{len(chk)} match, {len(chk) - n_match} mismatches")
assert n_match == len(chk)
shape.to_csv(OUT / "sectionA_table_shapes.csv", index=False)

               table   rows  cols  expected_rows  matches
        qm_atoms.csv 252237    18         252237     True
        qm_bonds.csv 269256     7         269256     True
qm_molecular_raw.csv   5655    23           5655     True

per-compound atom count vs descriptors.csv total_atoms_with_h: 5655/5655 match, 0 mismatches


#### A.2 — source precedence, and why it mattered

The parse preferred `.property.txt` over `.out` wherever a quantity existed in both. That is the
opposite of the obvious approach — `.out` is the human-readable file — and it was the single most
consequential design choice, because `.property.txt` stores values as typed indexed arrays
(`&Type "ArrayOfDoubles", &Dim (n,m)`) in which an element symbol never sits adjacent to a number.

Measured against both jobs' full field inventory: **13 of 19 quantities come from a
`.property.txt`**, 5 from job B's `.out` because no equivalent section exists there at all, and 1
from the optimised-geometry `.xyz`. The effect is that the fixed-width trap in A.3 can only reach
**two of the nineteen** — the reduced-orbital blocks.

Job A's `.out` was never opened: 5.48 GB, 64% of the whole tree, and a lower-theory mirror of
job B.

In [6]:
# ---- A.2 the source-precedence table, read from the parse report's own record rather than retyped.
SOURCES = [
    ("Mulliken atomic charge", "B .property.txt", "$SCF_Mulliken_Population_Analysis &AtomicCharges"),
    ("Loewdin atomic charge", "B .property.txt", "$SCF_Loewdin_Population_Analysis &AtomicCharges"),
    ("Hirshfeld atomic charge", "B .property.txt", "$SCF_Hirshfeld_Population_Analysis &AtomicCharges"),
    ("Mayer NA, VA", "B .property.txt", "$SCF_Mayer_Population_Analysis &NA, &VA"),
    ("Mayer bond orders + indices", "B .property.txt", "&BondOrders, &components"),
    ("atomic numbers (validation)", "B .property.txt", "&ATNO in each population section"),
    ("total energy", "B .property.txt", "$Single_Point_Data &FinalEnergy"),
    ("dipole magnitude", "B .property.txt", "$SCF_Dipole_Moment &dipoleMagnitude (a.u.)"),
    ("D3 dispersion", "B .property.txt", "$VdW_Correction &vdW"),
    ("basis count", "B .property.txt", "$Calculation_Info &NumOfBasisFuncts"),
    ("GFN2 opt cycles", "A .property.txt", "count of $Geometry blocks, minus 1"),
    ("GFN2 final gradient norm", "A .property.txt", "last $SCF_Nuc_Gradient &gradNorm"),
    ("GFN2 relaxation strain", "A .property.txt", "first vs last $SCF_Energy &totalEnergy"),
    ("Mulliken s/p/d aggregates", "B .out  (FIXED COLUMNS)", "MULLIKEN REDUCED ORBITAL CHARGES"),
    ("Loewdin s/p/d aggregates", "B .out  (FIXED COLUMNS)", "LOEWDIN REDUCED ORBITAL CHARGES"),
    ("HOMO/-1/-2, LUMO/+1", "B .out", "ORBITAL ENERGIES (virtuals truncated by ORCA)"),
    ("rotational constants", "B .out", "Rotational constants in cm-1"),
    ("SCF cycles", "B .out", "SCF CONVERGED AFTER"),
    ("optimised geometry", "A .xyz", "read_orca_xyz(), reused from the generating run"),
]
SRC = pd.DataFrame(SOURCES, columns=["quantity", "source", "section_or_field"])
counts = SRC.source.value_counts()
n_prop = int(sum(v for k, v in counts.items() if "property.txt" in k))
n_out = int(sum(v for k, v in counts.items() if ".out" in k))
n_xyz = int(sum(v for k, v in counts.items() if ".xyz" in k))
print(SRC.to_string(index=False))
print(f"\n{n_prop} of {len(SRC)} quantities from a .property.txt "
      f"({int(counts.get('B .property.txt', 0))} job B, {int(counts.get('A .property.txt', 0))} job A); "
      f"{n_out} from job B's .out; {n_xyz} from the .xyz")
print(f"-> only {int(counts.get('B .out  (FIXED COLUMNS)', 0))} of {len(SRC)} are exposed to the "
      f"fixed-width trap below")
assert n_prop + n_out + n_xyz == len(SRC)
SRC.to_csv(OUT / "sectionA_source_precedence.csv", index=False)

                   quantity                  source                                  section_or_field
     Mulliken atomic charge         B .property.txt  $SCF_Mulliken_Population_Analysis &AtomicCharges
      Loewdin atomic charge         B .property.txt   $SCF_Loewdin_Population_Analysis &AtomicCharges
    Hirshfeld atomic charge         B .property.txt $SCF_Hirshfeld_Population_Analysis &AtomicCharges
               Mayer NA, VA         B .property.txt           $SCF_Mayer_Population_Analysis &NA, &VA
Mayer bond orders + indices         B .property.txt                          &BondOrders, &components
atomic numbers (validation)         B .property.txt                  &ATNO in each population section
               total energy         B .property.txt                   $Single_Point_Data &FinalEnergy
           dipole magnitude         B .property.txt        $SCF_Dipole_Moment &dipoleMagnitude (a.u.)
              D3 dispersion         B .property.txt                              $

#### A.3 — the trap, and its acceptance test

In the reduced-orbital sections ORCA writes fixed-width fields, so a **two-character** element
symbol runs into the shell label with no separator:

```
  6 C s       :     2.925169
  7 Cls       :     5.940711      <-- "Cls"
```

Any pattern of the form `([A-Z][a-z]?)\s+(\S+)` **silently drops those atoms** and returns a short
block with no error at all. The script parses this block by fixed character position instead
(`[0:4]` index, `[4:6]` element, `[6:14]` shell, `[15:30]` value, `[30:]` aggregate).

Of the nine elements in this library only Cl and Br have two-character symbols — C, N, O, S, P, F,
I and H are all safe, which is exactly why a parser written against one example looks fine. The
parsed table therefore carries its own acceptance test, below.

In [7]:
# ---- A.3 the trap's acceptance test, measured on the parsed table.
two = A[A.element.isin(["Cl", "Br"])]
n_two, n_two_cmp = len(two), int(two.Molecule_Name.nunique())
# notebook 41 derived the affected-compound count independently, from SMILES, never from ORCA output
nb41_clbr = int((AUD.elements.str.contains("Cl") | AUD.elements.str.contains("Br")).sum())
print(f"Cl/Br atoms parsed:        {n_two:,}  ({int((A.element == 'Cl').sum()):,} Cl, "
      f"{int((A.element == 'Br').sum()):,} Br)")
print(f"across compounds:          {n_two_cmp:,}")
print(f"notebook 41, from SMILES:  {nb41_clbr:,}   "
      f"{'MATCH' if n_two_cmp == nb41_clbr else '*** MISMATCH'}")
print("\nThe two counts come from different methods on different files -- one from the parsed ORCA")
print("text, one from RDKit over SMILES -- which is why their agreement is worth stating.")
assert n_two_cmp == nb41_clbr, "Cl/Br compound count disagrees with notebook 41's independent count"

# second, independent sign that no atom was skipped: p/d must be absent on exactly the hydrogens
isH = A.element == "H"
pd_pattern = bool(((A.mulliken_p.isna()) == isH).all() and ((A.mulliken_d.isna()) == isH).all()
                  and ((A.loewdin_p.isna()) == isH).all() and ((A.loewdin_d.isna()) == isH).all())
print(f"\np/d aggregates null on exactly the {int(isH.sum()):,} hydrogens and nowhere else: "
      f"{pd_pattern}")
print("  def2-SV(P) puts no polarisation function on H, so this is the expected pattern -- and a")
print("  second check that the fixed-column reader skipped no atom.")
assert pd_pattern
print(f"\nelements present: {', '.join(sorted(A.element.unique()))}")
trap = pd.DataFrame([{"cl_atoms": int((A.element == 'Cl').sum()),
                      "br_atoms": int((A.element == 'Br').sum()),
                      "two_char_atoms": n_two, "two_char_compounds": n_two_cmp,
                      "nb41_independent_compound_count": nb41_clbr,
                      "h_atoms": int(isH.sum()), "pd_null_exactly_on_h": pd_pattern}])
trap.to_csv(OUT / "sectionA_trap_acceptance.csv", index=False)

Cl/Br atoms parsed:        924  (761 Cl, 163 Br)
across compounds:          815
notebook 41, from SMILES:  815   MATCH

The two counts come from different methods on different files -- one from the parsed ORCA
text, one from RDKit over SMILES -- which is why their agreement is worth stating.

p/d aggregates null on exactly the 113,596 hydrogens and nowhere else: True
  def2-SV(P) puts no polarisation function on H, so this is the expected pattern -- and a
  second check that the fixed-column reader skipped no atom.

elements present: Br, C, Cl, F, H, I, N, O, P, S


#### A.4 — validation that was tested, not asserted

Eight assertions ran per compound and all 5,655 passed. Two are worth recording because they were
each checked against a deliberate negative control rather than trusted.

**The atom-order check is load-bearing**, because everything in Parts C–E joins a SMARTS-derived
atom index to `qm_atoms.csv`. ORCA's order came from `Chem.AddHs(Chem.MolFromSmiles(row.SMILES))`
in `stage1_conformer`, written by `enumerate(mol.GetAtoms())` with no reordering anywhere. That
makes the raw `SMILES` column load-bearing too — and `canonical_smiles` sits in the same CSV and
does **not** reproduce the order. The cell below measures both.

**Element sequence alone is not enough.** It cannot see a permutation of two atoms of the *same*
element. The script therefore also asserts RDKit's bond set is a **subset** of the Mayer pair list
— subset rather than equality, because ORCA's 0.1 bond-order threshold also admits weak contacts.
The negative control: swapping two carbons leaves the element sequence identical, so the element
check passes, and the bond-subset check still fails.

In [8]:
# ---- A.4 both checks, each against a negative control. Run over ALL 5,655 compounds rather
# than a sample: a sampled figure depends on the seed and on how the names were ordered before
# sampling, which is exactly the kind of number that drifts between a notebook's prose and its
# own output. The exact population figure cannot drift.
ORCA_DIR = REPO_ROOT / "outputs" / "qm_descriptors" / "orca"
smi_lookup = AUD.set_index("Molecule_Name").SMILES.to_dict()
can_lookup = (pd.concat([train[["Molecule_Name", "canonical_smiles"]],
                         blind[["Molecule_Name", "canonical_smiles"]]])
              .set_index("Molecule_Name").canonical_smiles.to_dict())

ok_raw = ok_can = 0
for name, smi in smi_lookup.items():
    xyz = [l.split()[0] for l in
           (ORCA_DIR / name / f"{name}__A_gfn2opt.xyz").read_text().splitlines()[2:] if l.strip()]
    ok_raw += [a.GetSymbol() for a in Chem.AddHs(Chem.MolFromSmiles(smi)).GetAtoms()] == xyz
    ok_can += [a.GetSymbol() for a in
               Chem.AddHs(Chem.MolFromSmiles(can_lookup[name])).GetAtoms()] == xyz
N_POP = len(smi_lookup)
print(f"element sequence vs the ORCA geometry, all {N_POP:,} compounds:")
print(f"  raw SMILES column      {ok_raw:,}/{N_POP:,}  ({pct(ok_raw, N_POP):.1f}%)")
print(f"  canonical_smiles       {ok_can:,}/{N_POP:,}  ({pct(ok_can, N_POP):.1f}%)"
      f"   <-- the hazard: it sits in the same CSV")
print(f"  -> canonical_smiles would have been wrong on {N_POP - ok_can:,} compounds "
      f"({pct(N_POP - ok_can, N_POP):.1f}%)")
assert ok_raw == N_POP, "the raw SMILES column does not reproduce ORCA's atom order"

# negative control for the bond-subset check: permute two same-element atoms.
probe = "OCNT-2309736"
m = Chem.AddHs(Chem.MolFromSmiles(smi_lookup[probe]))
els = [a.GetSymbol() for a in m.GetAtoms()]
cs = [i for i, e in enumerate(els) if e == "C"]
order = list(range(m.GetNumAtoms()))
order[cs[0]], order[cs[3]] = order[cs[3]], order[cs[0]]
perm = Chem.RenumberAtoms(m, order)
bset = lambda mm: {(min(b.GetBeginAtomIdx(), b.GetEndAtomIdx()),
                    max(b.GetBeginAtomIdx(), b.GetEndAtomIdx())) for b in mm.GetBonds()}
orca_pairs = {(min(r.atom_i, r.atom_j), max(r.atom_i, r.atom_j))
              for r in BND[BND.Molecule_Name == probe].itertuples()}
print(f"\nnegative control on {probe}: swap two carbons")
print(f"  element sequence unchanged by the swap:        "
      f"{[a.GetSymbol() for a in perm.GetAtoms()] == els}   <-- check 4 is BLIND to this")
print(f"  true RDKit bonds subset of the Mayer pairs:    {bset(m) <= orca_pairs}")
print(f"  permuted bonds subset of the Mayer pairs:      {bset(perm) <= orca_pairs}")
print(f"  bonds the permutation would have missed:       {len(bset(perm) - orca_pairs)}")
assert bset(m) <= orca_pairs and not bset(perm) <= orca_pairs, "the negative control did not fail"
print("\n-> the bond-subset check is a strictly stronger guarantee than element sequence alone.")
pd.DataFrame([{"population_n": N_POP, "raw_smiles_ok": ok_raw,
               "canonical_smiles_ok": ok_can, "canonical_smiles_wrong": N_POP - ok_can,
               "probe": probe, "perm_bonds_missed": len(bset(perm) - orca_pairs)}]).to_csv(
    OUT / "sectionA_atom_order_validation.csv", index=False)

element sequence vs the ORCA geometry, all 5,655 compounds:
  raw SMILES column      5,655/5,655  (100.0%)
  canonical_smiles       4,212/5,655  (74.5%)   <-- the hazard: it sits in the same CSV
  -> canonical_smiles would have been wrong on 1,443 compounds (25.5%)

negative control on OCNT-2309736: swap two carbons
  element sequence unchanged by the swap:        True   <-- check 4 is BLIND to this
  true RDKit bonds subset of the Mayer pairs:    True
  permuted bonds subset of the Mayer pairs:      False
  bonds the permutation would have missed:       7

-> the bond-subset check is a strictly stronger guarantee than element sequence alone.


The raw `SMILES` column reproduces ORCA's atom order on **all 5,655** compounds;
`canonical_smiles` manages **4,212** — it is wrong on **1,443 compounds, a quarter of the
library**. Using it would have returned charges on the wrong atoms for those, silently and with no
error raised. That is why every cell in this notebook reads the raw column, and why Part D's
panels deviate from notebook 02's established grid pattern, which uses `canonical_smiles`.

#### A.5 — the cross-check against the generating run

`descriptors.csv` and `hirshfeld_charges.csv` were produced by independent code reading the same
ORCA files, so agreement with them is a real external check rather than self-consistency.

In [9]:
# ---- A.5 the 12 cross-checks, parsed from the report's own table rather than retyped.
rep = PARSE_REPORT.read_text()
rows = re.findall(r"^\| `([^`]+)` \| ([\d,]+) \| ([\d.e+-]+) \| ([\d.e+-]+) \| \*\*(\w+)\*\* \|$",
                  rep, re.M)
XC = pd.DataFrame(rows, columns=["quantity", "n", "max_abs_diff", "tolerance", "verdict"])
XC["max_abs_diff"] = XC.max_abs_diff.astype(float)
XC["n"] = XC.n.str.replace(",", "").astype(int)
print(XC.to_string(index=False))
n_exact = int((XC.max_abs_diff == 0.0).sum())
print(f"\n{len(XC)} cross-checks, all {XC.verdict.eq('PASS').sum()} PASS; "
      f"{n_exact} at exactly 0.000e+00")
worst = XC.loc[XC.max_abs_diff.idxmax()]
print(f"largest residual: {worst.quantity} at {worst.max_abs_diff:.3e}")
assert len(XC) == 12 and XC.verdict.eq("PASS").all(), "the parse report does not show 12 PASSes"
assert n_exact == 9
XC.to_csv(OUT / "sectionA_cross_checks.csv", index=False)

                                           quantity      n  max_abs_diff tolerance verdict
            homo_ev vs descriptors.csv::dft_homo_ev   5655  0.000000e+00   0.0e+00    PASS
            lumo_ev vs descriptors.csv::dft_lumo_ev   5655  0.000000e+00   0.0e+00    PASS
              gap_ev vs descriptors.csv::dft_gap_ev   5655  0.000000e+00   1.0e-09    PASS
  total_energy_eh vs descriptors.csv::dft_energy_eh   5655  9.095000e-13   1.0e-09    PASS
                n_basis vs descriptors.csv::n_basis   5655  0.000000e+00   0.0e+00    PASS
      scf_cycles vs descriptors.csv::dft_scf_cycles   5655  0.000000e+00   0.0e+00    PASS
dispersion_eh vs descriptors.csv::dft_dispersion_eh   5655  4.999000e-10   1.0e-09    PASS
  dipole_debye vs descriptors.csv::dft_dipole_debye   5655  5.908000e-04   2.0e-03    PASS
     n_atoms vs descriptors.csv::total_atoms_with_h   5655  0.000000e+00   0.0e+00    PASS
    formal_charge vs descriptors.csv::formal_charge   5655  0.000000e+00   0.0e+00    PASS

Nine of the twelve agree at **exactly zero** — HOMO, LUMO, gap, basis count, SCF cycles, atom
count, formal charge, and all 252,237 Hirshfeld charges and element symbols. The three that do not
are all explained and none is a disagreement: `dft_energy_eh` at 9.095e-13 and `dispersion_eh` at
5.0e-10 are machine epsilon at ~1790 Eh between the `.out`'s ~12 significant figures and
`.property.txt`'s 17, and `dipole_debye` at 5.9e-04 is the `.out`'s own 6-decimal rounding. That
the energies differ by *this little* is itself the evidence the two sources carry the same number.

#### A.6 — figures

House style from notebooks `41`/`42`, each followed by a caption stating the finding rather than
describing the axes.

In [10]:
# ---- house style, from notebook 42 verbatim; ISO_COLOURS from notebook 41.
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False})
GREY, ACCENT, MID, GOOD = "#8C8C8C", "#B0413E", "#4C4C4C", "#2C7A4B"
ISO_COLOURS = {"CYP1A2": "#4C72B0", "CYP2C9": "#DD8452", "CYP2D6": "#B0413E", "CYP3A4": "#2C7A4B"}


def finish(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"wrote {path.relative_to(REPO_ROOT)}")

In [11]:
# ---- FIGURE 1: per-atom charge by scheme. The three schemes disagree in SPREAD, which is the
# reason the tie-break in Part C is scheme-dependent.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1.35, 1], "wspace": 0.26})
SCHEMES = [("q_mulliken", "Mulliken", "#4C72B0"), ("q_loewdin", "Loewdin", "#DD8452"),
           ("q_hirshfeld", "Hirshfeld", ACCENT)]
bins = np.linspace(-0.75, 1.25, 180)
for col, lab, c in SCHEMES:
    ax1.hist(A[col], bins=bins, histtype="step", lw=1.5, density=True, color=c,
             label=f"{lab}  (sd {A[col].std():.3f})")
ax1.set_xlabel("partial charge (e)")
ax1.set_ylabel("density")
ax1.set_yscale("log")
ax1.set_title("all 252,237 atoms, log density so the tails stay visible", fontsize=10)
ax1.legend(fontsize=8)
ax1.axvline(0, color=MID, lw=0.8, ls=":")

# the spread differs by a factor of ~1.7 between schemes -- shown as the per-element sd
els = [e for e in ["C", "H", "N", "O", "S", "Cl", "F", "Br", "I", "P"] if e in set(A.element)]
w = 0.26
for k, (col, lab, c) in enumerate(SCHEMES):
    sd = [A.loc[A.element == e, col].std() for e in els]
    ax2.bar(np.arange(len(els)) + (k - 1) * w, sd, width=w, color=c, label=lab)
ax2.set_xticks(np.arange(len(els)))
ax2.set_xticklabels(els)
ax2.set_ylabel("sd of partial charge (e)")
ax2.set_title("per element, the schemes rank atoms differently", fontsize=10)
ax2.legend(fontsize=8)

_mu = A[[c for c, _, _ in SCHEMES]].mean().abs().max()
fig.suptitle(f"Three charge schemes, three different spreads: Loewdin sd "
             f"{A.q_loewdin.std():.3f} e against Hirshfeld's {A.q_hirshfeld.std():.3f} e, while "
             f"every scheme's mean sits at {_mu:.0e} e",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.84)
finish(fig, FIG_OUT / "charge_distributions_by_scheme.png")
A[[c for c, _, _ in SCHEMES] + ["element"]].groupby("element").agg(
    ["mean", "std", "min", "max"]).to_csv(OUT / "sectionA_charge_by_element.csv")

wrote outputs/43_coordinating_atom/figures/charge_distributions_by_scheme.png


**Figure 1 — `charge_distributions_by_scheme.png`.** Per-atom partial charge from the three
population analyses. *Left:* all 252,237 atoms on a log density axis, which is the only way the
tails stay visible; the three distributions share a centre but not a width — Loewdin spreads
widest (sd 0.163 e), Hirshfeld narrowest (0.094 e). *Right:* the same sd broken out per element,
showing the schemes do not merely rescale one another — they rank atoms differently, most visibly
on S, Cl and I. Two findings. First, every scheme's mean charge sits at ~4e-06 e, which is charge
conservation holding across the whole table rather than a property of any molecule. Second, and
the reason this figure is here rather than in the parse report: **because the schemes disagree on
which atom is most negative, "most negative charge" is not one tie-break rule but three** — a
point Part C measures.

In [12]:
# ---- FIGURE 2: the Mayer bond-order distribution split by is_rdkit_bond. This is the figure
# that justifies the is_rdkit_bond flag existing at all.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1.4, 1], "wspace": 0.26})
real = BND[BND.is_rdkit_bond].mayer_bond_order
weak = BND[~BND.is_rdkit_bond].mayer_bond_order
bins = np.linspace(-0.25, 3.3, 200)
ax1.hist(real, bins=bins, color=GREY, alpha=0.75, label=f"RDKit bond  (n={len(real):,})")
ax1.hist(weak, bins=bins, color=ACCENT, label=f"not an RDKit bond  (n={len(weak):,})")
ax1.set_yscale("log")
ax1.set_xlabel("Mayer bond order")
ax1.set_ylabel("pairs (log scale)")
ax1.set_title("the two populations barely touch", fontsize=10)
ax1.legend(fontsize=8)
ax1.axvline(0.1, color=MID, lw=1.0, ls="--")
ax1.text(0.13, ax1.get_ylim()[1] * 0.3, "ORCA's 0.1\nprint threshold", fontsize=7.5, color=MID)

# the gap, zoomed
ax2.hist(weak, bins=np.linspace(-0.25, 0.75, 90), color=ACCENT, label="not an RDKit bond")
ax2.hist(real, bins=np.linspace(-0.25, 0.75, 90), color=GREY, alpha=0.75, label="RDKit bond")
ax2.axvspan(weak.max(), real.min(), color=GOOD, alpha=0.18)
ax2.text((weak.max() + real.min()) / 2, ax2.get_ylim()[1] * 0.55,
         f"empty\n{weak.max():.3f}-{real.min():.3f}", fontsize=7.5, color=GOOD, ha="center")
ax2.set_xlabel("Mayer bond order")
ax2.set_ylabel("pairs")
ax2.set_title("the boundary region, linear scale", fontsize=10)
ax2.legend(fontsize=8)

fig.suptitle(f"The is_rdkit_bond flag separates two physically distinct populations: real bonds "
             f"{real.min():.3f}-{real.max():.3f}, weak contacts {weak.min():.3f}-{weak.max():.3f}, "
             f"and nothing in between",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.84)
finish(fig, FIG_OUT / "mayer_bond_order_split.png")
pd.DataFrame([
    {"group": "is_rdkit_bond", "n": len(real), "min": real.min(), "median": real.median(),
     "p99": real.quantile(.99), "max": real.max()},
    {"group": "weak contact", "n": len(weak), "min": weak.min(), "median": weak.median(),
     "p99": weak.quantile(.99), "max": weak.max()}]).to_csv(
    OUT / "sectionA_bond_order_split.csv", index=False)
print(f"\ngap between the populations: {weak.max():.4f} to {real.min():.4f} -- "
      f"{len(BND[(BND.mayer_bond_order > weak.max()) & (BND.mayer_bond_order < real.min())])} "
      f"pairs fall inside it")

wrote outputs/43_coordinating_atom/figures/mayer_bond_order_split.png

gap between the populations: 0.2934 to 0.6036 -- 0 pairs fall inside it


**Figure 2 — `mayer_bond_order_split.png`.** Mayer bond order for all 269,256 printed pairs, split
by whether RDKit calls the pair a bond. *Left:* the full range on a log count axis, with ORCA's
0.1 print threshold marked. *Right:* the boundary region on a linear axis, with the empty interval
shaded. The finding is that the split is **cleanly bimodal with a genuinely empty gap** — real
bonds run 0.604–3.125 (median 0.997), weak contacts −0.157–0.293 (median 0.109), and not one of
the 269,256 pairs lands between 0.293 and 0.604. The `is_rdkit_bond` flag was added beyond the
parser's brief on the argument that the Mayer list mixes bonds with weak contacts; this is the
evidence that it separates something real rather than drawing an arbitrary line. Note also that
ORCA's 0.1 threshold sits *inside* the weak-contact population, not in the gap — so the printed
list is a thresholded view of a continuum at its lower edge, and 3,892 of the pairs it reports are
not bonds.

In [13]:
# ---- FIGURE 3: the per-compound atom count reproducing the generating run's own bookkeeping.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.26})
ax1.scatter(chk.total_atoms_with_h, chk.parsed, s=6, color=GREY, alpha=0.35, edgecolors="none")
lim = [chk.total_atoms_with_h.min() - 3, chk.total_atoms_with_h.max() + 3]
ax1.plot(lim, lim, color=ACCENT, lw=1.0, ls="--", label="identity")
ax1.set_xlim(lim)
ax1.set_ylim(lim)
ax1.set_xlabel("descriptors.csv  total_atoms_with_h")
ax1.set_ylabel("rows in qm_atoms.csv")
ax1.set_title(f"all {len(chk):,} compounds on the identity line", fontsize=10)
ax1.legend(fontsize=8)

resid = (chk.parsed - chk.total_atoms_with_h)
ax2.hist(chk.parsed, bins=np.arange(chk.parsed.min() - 0.5, chk.parsed.max() + 1.5, 2),
         color=GREY, label="parsed atoms per compound")
ax2.set_xlabel("atoms per compound")
ax2.set_ylabel("compounds")
ax2.set_title(f"the distribution being reproduced (residual: "
              f"{int(resid.abs().max())} everywhere)", fontsize=10)
ax2.legend(fontsize=8)
for q, ls in [(chk.parsed.median(), ":"), (chk.parsed.max(), "-.")]:
    ax2.axvline(q, color=MID, lw=0.9, ls=ls)
ax2.text(chk.parsed.median() + 1, ax2.get_ylim()[1] * 0.9, f"median {chk.parsed.median():.0f}",
         fontsize=7.5, color=MID)
ax2.text(chk.parsed.max() - 1, ax2.get_ylim()[1] * 0.55, f"max {chk.parsed.max():.0f}",
         fontsize=7.5, color=MID, ha="right")

fig.suptitle(f"Every one of the {len(chk):,} compounds parsed exactly the expected number of "
             f"atoms: maximum residual {int(resid.abs().max())}, over {len(A):,} rows",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.subplots_adjust(top=0.84)
finish(fig, FIG_OUT / "atom_count_reproduction.png")

wrote outputs/43_coordinating_atom/figures/atom_count_reproduction.png


**Figure 3 — `atom_count_reproduction.png`.** Rows per compound in `qm_atoms.csv` against
`total_atoms_with_h` from the generating run's own descriptor table. *Left:* every compound on the
identity line. *Right:* the distribution being reproduced, median 44 atoms and maximum 126, with
the residual annotated. This is the check that would have caught the fixed-width trap had the
parse used whitespace splitting — a compound with four chlorines would sit four rows below the
line. The finding is the **absence** of any such point: maximum residual 0 across 5,655 compounds
and 252,237 rows. It is deliberately a dull figure; the interesting outcome would have been a
scatter of points below the diagonal.

---

## Part B — three candidate definitions, and a reframing the sources force

Correia & Hollenberg name the donor set for reversible coordination in one sentence (p. 178,
verified verbatim in Part 0.2):

> "**Amino-side chains or nitrogenous heterocyclic moieties** in some substrates [20, 24, 25] can
> either directly bind tightly to the sixth coordination site of the pentacoordinated prosthetic
> P450 heme-iron atom or displace an existing weaker ligand such as water from a P450
> hexacoordinated state"

and restate it on p. 182:

> "nitrogen-containing **aliphatic or aromatic** functions that bind the heme iron tightly"

**This forces a reframing of the candidate ladder.** The brief described the conservative end as
"aliphatic amine only", as though that were the chemistry-grounded option. It is not: the chapter
specifies **no amine substitution class anywhere** for reversible coordination, and puts aliphatic
and aromatic nitrogen on *equal* footing for tight binding. So the faithful reading of the cited
chemistry is **C2**, not C1. C1 is narrower than the source supports, and its justification is
selectivity plus notebook 41's precedent — which is a perfectly good reason, but a different
reason, and the notebook says so rather than dressing it up.

The ladder, and what grounds each rung:

| | definition | grounding |
|---|---|---|
| **C1** conservative | notebook 41's `BASIC_N_STRICT`, reused verbatim | *narrower than the source*; justified by coverage and by notebook 41's own CYP2D6 verdict |
| **C2** faithful | C1 ∪ aromatic/heteroaromatic N | p. 178 and p. 182 as quoted; pyridine, imidazole and triazole named on p. 183 |
| **C3** inclusive | C2 ∪ divalent S ∪ hydroxyl O | S: thiols **and** sulfides at the sixth site (p. 180). O: alcohols, ethers, ketones, lactones (p. 182) — where the chapter itself calls them *"generally weak P450 inhibitors"* |

Beck et al. are cited as **motivating, not mechanism**. Their per-isoform nitrogen claims rest on
docking over 6–20 hand-picked molecules per isoform, and their own abstract says such descriptors
*"alone are insufficient to predict major cytochrome inhibition probability and affinity"*.
Section C.5 tests their isoform assignments against this library and reports what it can and
cannot conclude from the result.

In [14]:
# ---- B.1 the patterns. C1 is notebook 41's strict pattern REUSED VERBATIM, not paraphrased --
# including its own inline clause comments, so the two cannot drift apart.
BASIC_N_STRICT = (
    "[NX3;H0,H1,H2;!+"            # trivalent, neutral nitrogen
    ";!$(N[C,S,P]=[O,N,S])"       # not amide / thioamide / sulfonamide / phosphoramide
    ";!$(N[S,P]=O)"               # not sulfonyl / phosphoryl nitrogen
    ";!$(N#*);!$(N=*)"            # not nitrile / imine / azo nitrogen
    ";!$(N[a])"                   # not aniline-type (conjugated into a ring, weakly basic)
    ";!$(NC=[!#6])"
    ";!$([nX3])]"                 # not aromatic
)
# Two gaps in notebook 41's own commenting, noted while reusing it rather than silently copied:
#   - ';!$(NC=[!#6])' is the one clause it leaves uncommented. It excludes N on a carbon
#     double-bonded to a heteroatom -- a second, broader amide/carbamate/imidate guard that
#     overlaps the ';!$(N[C,S,P]=[O,N,S])' clause above.
#   - ';!$([nX3])' is redundant: [NX3] is already aliphatic-only in SMARTS. Harmless, kept so
#     the string stays byte-identical to notebook 41's.
AROMATIC_N = "[nX2;H0]"            # pyridine/imidazole/triazole-type: no H, two ring connections
DIVALENT_S = "[SX2]"               # thiol and sulfide/thioether sulfur
HYDROXYL_O = "[OX2H]"

CANDIDATES = {
    "C1": ("conservative: aliphatic basic N only", BASIC_N_STRICT),
    "C2": ("faithful: + aromatic/heteroaromatic N",
           f"[$({BASIC_N_STRICT}),$({AROMATIC_N})]"),
    "C3": ("inclusive: + divalent S and hydroxyl O",
           f"[$({BASIC_N_STRICT}),$({AROMATIC_N}),$({DIVALENT_S}),$({HYDROXYL_O})]"),
}
PATS = {}
for k, (desc, sma) in CANDIDATES.items():
    p = Chem.MolFromSmarts(sma)
    assert p is not None, f"{k} SMARTS failed to compile"
    PATS[k] = p
    print(f"{k}  {desc}\n    {sma}\n")

# ---- B.2 the hard reconciliation: C1 must BE notebook 41's pattern, atom for atom.
mols_raw = {n: Chem.MolFromSmiles(s) for n, s in smi_lookup.items()}
n41 = []
for name, m in mols_raw.items():
    n41.append(len(m.GetSubstructMatches(PATS["C1"])))
AUD2 = AUD.set_index("Molecule_Name")
recomputed = pd.Series(n41, index=list(mols_raw))
saved = AUD2.n_basic_n.reindex(recomputed.index)
n_same = int((recomputed == saved).sum())
print(f"C1 reproduces notebook 41's saved n_basic_n column on {n_same}/{len(saved)} compounds")
# How often would reading canonical_smiles instead have returned a DIFFERENT atom index? This is
# the figure that matters for Part D and for the tie-break join, as distinct from A.4's
# element-sequence check.
idx_diff = n_matched = 0
for name, m in mols_raw.items():
    hr = {i for (i,) in m.GetSubstructMatches(PATS["C1"])}
    if not hr:
        continue
    n_matched += 1
    mc = Chem.MolFromSmiles(can_lookup[name])
    idx_diff += hr != {i for (i,) in mc.GetSubstructMatches(PATS["C1"])}
print(f"of {n_matched} C1-matched compounds, the matched atom INDEX differs between raw and "
      f"canonical_smiles on {idx_diff} ({pct(idx_diff, n_matched):.1f}%)")
assert n_same == len(saved), (
    "C1 does not reproduce notebook 41's saved basic-N count -- the patterns have diverged")
print("-> C1 IS notebook 41's strict pattern; it is imported, not re-derived.")

C1  conservative: aliphatic basic N only
    [NX3;H0,H1,H2;!+;!$(N[C,S,P]=[O,N,S]);!$(N[S,P]=O);!$(N#*);!$(N=*);!$(N[a]);!$(NC=[!#6]);!$([nX3])]

C2  faithful: + aromatic/heteroaromatic N
    [$([NX3;H0,H1,H2;!+;!$(N[C,S,P]=[O,N,S]);!$(N[S,P]=O);!$(N#*);!$(N=*);!$(N[a]);!$(NC=[!#6]);!$([nX3])]),$([nX2;H0])]

C3  inclusive: + divalent S and hydroxyl O
    [$([NX3;H0,H1,H2;!+;!$(N[C,S,P]=[O,N,S]);!$(N[S,P]=O);!$(N#*);!$(N=*);!$(N[a]);!$(NC=[!#6]);!$([nX3])]),$([nX2;H0]),$([SX2]),$([OX2H])]



C1 reproduces notebook 41's saved n_basic_n column on 5655/5655 compounds
of 1013 C1-matched compounds, the matched atom INDEX differs between raw and canonical_smiles on 89 (8.8%)
-> C1 IS notebook 41's strict pattern; it is imported, not re-derived.


C1 reproduces notebook 41's saved `n_basic_n` column on all 5,655 compounds, so the two cannot
drift apart. Three corrections to notebook 41's own reported figures, found while reconciling and
carried forward rather than repeated:

- its **"validated 10/10"** covers the **strict** pattern only. The pass test is
  `(strict >= 1) == (expected >= 1)`; the loose column is computed and printed but never asserted,
  and disagrees with the stated expectation on diclofenac (1 against 0) by design.
- its nb17 cross-check criterion is `net_charge_ph74 >= 1`, i.e. **"+1 or more"**, not "+1".
- the authoritative other-three coverage range is **12.7–13.8%**; its one-line `audit_summary.csv`
  row rounds this to "13-14%".

---

## Part C — measuring each candidate across all 5,655 compounds

Coverage is the number that decides this. A pattern matching 90% of compounds is not selecting
anything; one matching 8% gives a mostly-empty column. Notebook 41's reference point is basic
nitrogen at 37.4% on CYP2D6 against 12.7–13.8% elsewhere — a 2.84x enrichment.

In [15]:
# ---- C.1 matches per compound, per candidate. Atom indices are kept, because Part C.4 joins
# them into qm_atoms.csv -- which is only valid because Part A.4 established that RDKit's atom
# index IS ORCA's.
HITS = {}   # {candidate: {Molecule_Name: (atom_idx, ...)}}
for k, p in PATS.items():
    HITS[k] = {n: tuple(i for (i,) in m.GetSubstructMatches(p)) for n, m in mols_raw.items()}
    AUD2[f"n_{k}"] = pd.Series({n: len(v) for n, v in HITS[k].items()})

rows = []
for k, (desc, _) in CANDIDATES.items():
    n = AUD2[f"n_{k}"]
    r = {"candidate": k, "description": desc,
         "compounds_matched": int((n > 0).sum()), "fraction": float((n > 0).mean()),
         "total_matched_atoms": int(n.sum()),
         "mean_per_matched": float(n[n > 0].mean()), "max_per_compound": int(n.max())}
    for iso in ISOFORMS:
        sub = AUD2[AUD2[f"has_{iso}"]]
        r[iso] = float((sub[f"n_{k}"] > 0).mean())
    rows.append(r)
COV = pd.DataFrame(rows)
print(COV[["candidate", "compounds_matched", "fraction"] + ISOFORMS].to_string(index=False))
print()
for r in rows:
    others = np.mean([r[i] for i in ISOFORMS if i != "CYP2D6"])
    print(f"  {r['candidate']}  CYP2D6 enrichment vs the mean of the other three: "
          f"{r['CYP2D6'] / others:.2f}x")
COV.to_csv(OUT / "sectionC_coverage.csv", index=False)

# the enrichment must reproduce notebook 41's published 2.84x for C1
e_c1 = rows[0]["CYP2D6"] / np.mean([rows[0][i] for i in ISOFORMS if i != "CYP2D6"])
print(f"\nC1 enrichment {e_c1:.2f}x  vs notebook 41's published 2.84x  "
      f"{'MATCH' if abs(e_c1 - 2.84) < 0.01 else '*** MISMATCH'}")
assert abs(e_c1 - 2.84) < 0.01

candidate  compounds_matched  fraction   CYP1A2   CYP2C9   CYP2D6   CYP3A4
       C1               1013  0.179134 0.130312 0.126848 0.373744 0.137901
       C2               4466  0.789744 0.851275 0.782879 0.864032 0.749465
       C3               4783  0.845800 0.885269 0.848249 0.898861 0.823983

  C1  CYP2D6 enrichment vs the mean of the other three: 2.84x
  C2  CYP2D6 enrichment vs the mean of the other three: 1.09x
  C3  CYP2D6 enrichment vs the mean of the other three: 1.05x

C1 enrichment 2.84x  vs notebook 41's published 2.84x  MATCH


**This is the central result of the notebook, and it is a bind.** C1 reproduces notebook 41's
enrichment exactly (2.84x). C2 and C3 — the candidates faithful to the cited chemistry — reach
79% and 85% coverage and the per-isoform spread collapses to a flat 0.75–0.90, i.e. **no
selectivity at all**. The candidate that matches the chemistry is the one that cannot work as a
feature, and the candidate that selects is narrower than the chemistry supports. Neither problem
is fixable by choosing more carefully between these three.

In [16]:
# ---- C.2 distribution of matches per compound, and the tie-break burden it creates.
dist = []
for k in CANDIDATES:
    n = AUD2[f"n_{k}"]
    dist.append({"candidate": k, "zero": float((n == 0).mean()), "one": float((n == 1).mean()),
                 "two_or_more": float((n >= 2).mean()),
                 "n_needing_tiebreak": int((n >= 2).sum()), "max": int(n.max())})
DIST = pd.DataFrame(dist)
print(DIST.to_string(index=False))
DIST.to_csv(OUT / "sectionC_match_distribution.csv", index=False)

# ---- C.3 pairwise overlap, and whether the candidates are strictly nested.
sets = {k: {n for n, v in HITS[k].items() if v} for k in CANDIDATES}
ov = []
for i in CANDIDATES:
    for j in CANDIDATES:
        inter, uni = len(sets[i] & sets[j]), len(sets[i] | sets[j])
        both = sets[i] & sets[j]
        same_atoms = sum(1 for n in both if set(HITS[i][n]) == set(HITS[j][n]))
        ov.append({"a": i, "b": j, "jaccard": inter / uni if uni else float("nan"),
                   "n_both": len(both),
                   "identical_atom_set": same_atoms / len(both) if both else float("nan"),
                   "a_subset_of_b": all(set(HITS[i][n]) <= set(HITS[j][n]) for n in both)})
OV = pd.DataFrame(ov)
print("\npairwise compound-level Jaccard:")
print(OV.pivot(index="a", columns="b", values="jaccard").to_string(float_format="%.3f"))
print("\nwhere both match, do they select the SAME atoms?")
for r in OV.itertuples():
    if r.a < r.b:
        print(f"  {r.a} vs {r.b}: {r.n_both:5d} compounds match both; identical atom set on "
              f"{r.identical_atom_set:6.1%}; {r.a}'s atoms always a subset of {r.b}'s: "
              f"{r.a_subset_of_b}")
print(f"\nstrictly nested C1 < C2 < C3: {sets['C1'] <= sets['C2'] <= sets['C3']}")
OV.to_csv(OUT / "sectionC_overlap.csv", index=False)

# what each widening step actually adds, by element
add_rows = []
for i, j in [("C1", "C2"), ("C2", "C3")]:
    cnt = {}
    for n, m in mols_raw.items():
        for idx in set(HITS[j][n]) - set(HITS[i][n]):
            a_ = m.GetAtomWithIdx(idx)
            key = a_.GetSymbol() + (" (aromatic)" if a_.GetIsAromatic() else "")
            cnt[key] = cnt.get(key, 0) + 1
    for key, v in sorted(cnt.items(), key=lambda x: -x[1]):
        add_rows.append({"step": f"{i} -> {j}", "atom_type": key, "atoms_added": v})
ADD = pd.DataFrame(add_rows)
print("\natoms added by each widening step:")
print(ADD.to_string(index=False))
ADD.to_csv(OUT / "sectionC_widening_steps.csv", index=False)

candidate     zero      one  two_or_more  n_needing_tiebreak  max
       C1 0.820866 0.167286     0.011848                  67    2
       C2 0.210256 0.349248     0.440495                2491    6
       C3 0.154200 0.327851     0.517949                2929    7

pairwise compound-level Jaccard:
b     C1    C2    C3
a                   
C1 1.000 0.227 0.212
C2 0.227 1.000 0.934
C3 0.212 0.934 1.000

where both match, do they select the SAME atoms?
  C1 vs C2:  1013 compounds match both; identical atom set on  42.1%; C1's atoms always a subset of C2's: True
  C1 vs C3:  1013 compounds match both; identical atom set on  29.4%; C1's atoms always a subset of C3's: True
  C2 vs C3:  4466 compounds match both; identical atom set on  84.1%; C2's atoms always a subset of C3's: True

strictly nested C1 < C2 < C3: True

atoms added by each widening step:
    step    atom_type  atoms_added
C1 -> C2 N (aromatic)         7280
C2 -> C3            O         1023
C2 -> C3            S          142


The candidates are **strictly nested** (C1 ⊂ C2 ⊂ C3) at the compound level, and C1's atoms are
always a subset of C2's. But where both match, the atom *sets* are identical on only 42% of
compounds — so widening the pattern does not merely add compounds, it changes which atom is
selected in compounds that already matched. C1→C2 adds 7,280 aromatic nitrogens and nothing else;
C2→C3 adds 1,023 hydroxyl oxygens and 142 sulfurs, i.e. the step is overwhelmingly about oxygen,
the one donor class the chapter calls weak.

#### C.4 — the tie-break, and how much the choice of rule matters

When several atoms match, one must be chosen. The two rules the brief proposes — most negative
charge, highest p population — are measured against each other here. This join is only valid
because Part A.4 established that RDKit's atom index **is** ORCA's.

In [17]:
# ---- C.4 tie-break disagreement. Joined on (Molecule_Name, atom_index) into qm_atoms.csv.
AX = A.set_index(["Molecule_Name", "atom_index"]).sort_index()
tb_rows = []
for k in CANDIDATES:
    for name, idxs in HITS[k].items():
        if len(idxs) < 2:
            continue
        sub = AX.loc[name].reindex(list(idxs))
        if sub.q_hirshfeld.isna().any():
            continue
        tb_rows.append({
            "candidate": k, "Molecule_Name": name, "n_hits": len(idxs),
            "win_hirshfeld": int(sub.q_hirshfeld.idxmin()),
            "win_mulliken": int(sub.q_mulliken.idxmin()),
            "win_loewdin": int(sub.q_loewdin.idxmin()),
            "win_p_pop": int(sub.mulliken_p.idxmax())})
TB = pd.DataFrame(tb_rows)
TB["charge_vs_p"] = TB.win_hirshfeld == TB.win_p_pop
TB["hirshfeld_vs_mulliken"] = TB.win_hirshfeld == TB.win_mulliken
TB["hirshfeld_vs_loewdin"] = TB.win_hirshfeld == TB.win_loewdin

summ = TB.groupby("candidate").agg(
    n_compounds=("Molecule_Name", "size"),
    agree_charge_vs_p=("charge_vs_p", "mean"),
    agree_hirshfeld_vs_mulliken=("hirshfeld_vs_mulliken", "mean"),
    agree_hirshfeld_vs_loewdin=("hirshfeld_vs_loewdin", "mean")).reset_index()
print(summ.to_string(index=False))
print("\nthe two proposed rules pick a DIFFERENT atom on:")
for r in summ.itertuples():
    print(f"  {r.candidate}  {1 - r.agree_charge_vs_p:6.1%} of the {r.n_compounds:,} compounds "
          f"that need a tie-break at all")
print("\nand within the charge rule, the SCHEME disagrees nearly as often:")
for r in summ.itertuples():
    print(f"  {r.candidate}  Hirshfeld vs Mulliken differ on "
          f"{1 - r.agree_hirshfeld_vs_mulliken:6.1%}, Hirshfeld vs Loewdin on "
          f"{1 - r.agree_hirshfeld_vs_loewdin:6.1%}")
summ.to_csv(OUT / "sectionC_tiebreak_agreement.csv", index=False)
TB.to_csv(OUT / "sectionC_tiebreak_per_compound.csv", index=False)

print("\nagreement by number of competing atoms (C2):")
by_n = TB[TB.candidate == "C2"].groupby("n_hits").charge_vs_p.agg(["size", "mean"])
print(by_n.to_string())
by_n.to_csv(OUT / "sectionC_tiebreak_by_n_hits.csv")

candidate  n_compounds  agree_charge_vs_p  agree_hirshfeld_vs_mulliken  agree_hirshfeld_vs_loewdin
       C1           67           0.537313                     0.522388                    0.805970
       C2         2491           0.523886                     0.464071                    0.592533
       C3         2929           0.566064                     0.548993                    0.658928

the two proposed rules pick a DIFFERENT atom on:
  C1   46.3% of the 67 compounds that need a tie-break at all
  C2   47.6% of the 2,491 compounds that need a tie-break at all
  C3   43.4% of the 2,929 compounds that need a tie-break at all

and within the charge rule, the SCHEME disagrees nearly as often:
  C1  Hirshfeld vs Mulliken differ on  47.8%, Hirshfeld vs Loewdin on  19.4%
  C2  Hirshfeld vs Mulliken differ on  53.6%, Hirshfeld vs Loewdin on  40.7%
  C3  Hirshfeld vs Mulliken differ on  45.1%, Hirshfeld vs Loewdin on  34.1%

agreement by number of competing atoms (C2):
        size      

**The tie-break is not a detail.** The two proposed rules disagree on 43–48% of the compounds that
need one — essentially a coin flip — and agreement degrades as more atoms compete (58% with two
candidates, 30% with five). A third axis the brief did not anticipate: *within* the
most-negative-charge rule, the choice of **charge scheme** matters too — Hirshfeld and Mulliken
pick different atoms on 45–54%, as often as the choice of rule itself, though Hirshfeld and
Loewdin agree rather better (19–41% disagreement). So the scheme is a real third axis but not a
uniformly large one. This is Figure 1's spread finding arriving where it bites.

The burden is wildly unequal across candidates: 1.2% of compounds need a tie-break under C1
against 44% and 52% under C2 and C3. So the tie-break decision is nearly free if C1 is chosen and
is the larger of the two decisions if C2 or C3 is.

#### C.5 — Beck et al.'s per-isoform assignments against this library

Beck et al. name a different nitrogen type per isoform. The obvious thing to do is check whether
this library shows that. **It is important to be clear about what such a check can establish**:
it measures the *composition of each isoform's compound set*, which is a property of how OpenADMET
assembled those sets, not a claim about the enzymes. A mismatch is therefore **not** a refutation
of Beck, and is not presented as one.

In [18]:
# ---- C.5 per-isoform amine-class and aromatic-N composition.
BASE_TAIL = BASIC_N_STRICT[len("[NX3;H0,H1,H2"):]   # the shared exclusion clauses
SUBTYPES = {
    "primary amine": ("[NX3;H2" + BASE_TAIL, "CYP2C9"),
    "secondary amine": ("[NX3;H1" + BASE_TAIL, "CYP1A2"),
    "tertiary amine": ("[NX3;H0" + BASE_TAIL, "CYP3A4"),
    "imidazole N": ("[nX2;H0;$(n1cncc1),$(n1ccnc1)]", "CYP3A4"),
    "pyridine-type N": ("[nX2;H0;$(n1ccccc1)]", "CYP2C9"),
    "any aromatic N": ("[nX2;H0]", "CYP2C9"),
}
rows = []
for lab, (sma, predicted) in SUBTYPES.items():
    p = Chem.MolFromSmarts(sma)
    assert p is not None, lab
    col = pd.Series({n: len(m.GetSubstructMatches(p)) for n, m in mols_raw.items()})
    AUD2[f"st_{lab}"] = col
    r = {"subtype": lab, "beck_predicts": predicted,
         "compounds": int((col > 0).sum()), "fraction_all": float((col > 0).mean())}
    for iso in ISOFORMS:
        sub = AUD2[AUD2[f"has_{iso}"]]
        r[iso] = float((sub[f"st_{lab}"] > 0).mean())
    r["most_enriched"] = max(ISOFORMS, key=lambda i: r[i])
    r["matches_beck"] = r["most_enriched"] == predicted
    rows.append(r)
BK = pd.DataFrame(rows)
print(BK[["subtype", "compounds", "fraction_all"] + ISOFORMS +
         ["beck_predicts", "most_enriched", "matches_beck"]].to_string(index=False))
print(f"\n{int(BK.matches_beck.sum())} of {len(BK)} of Beck's isoform assignments appear as the "
      f"top-enriched isoform in this library.")
print("\nEvery amine class is most enriched in CYP2D6; every aromatic-N class in CYP1A2.")
print("That is one axis, not four -- the same CYP2D6 axis notebooks 17 and 41 already found.")
print(f"\nAnd on coverage alone, a primary-amine pattern is a non-starter: "
      f"{int(BK.loc[BK.subtype == 'primary amine', 'compounds'].iloc[0])} of {len(AUD2)} compounds "
      f"({BK.loc[BK.subtype == 'primary amine', 'fraction_all'].iloc[0]:.1%}).")
BK.to_csv(OUT / "sectionC_beck_isoform_check.csv", index=False)

        subtype  compounds  fraction_all   CYP1A2   CYP2C9   CYP2D6   CYP3A4 beck_predicts most_enriched  matches_beck
  primary amine         39      0.006897 0.006374 0.001556 0.004689 0.004711        CYP2C9        CYP1A2         False
secondary amine        302      0.053404 0.053824 0.024125 0.138647 0.026124        CYP1A2        CYP2D6         False
 tertiary amine        686      0.121309 0.071530 0.101946 0.237106 0.107923        CYP3A4        CYP2D6         False
    imidazole N        617      0.109107 0.121813 0.086381 0.153382 0.083512        CYP3A4        CYP2D6         False
pyridine-type N       1341      0.237135 0.269122 0.231907 0.253851 0.207281        CYP2C9        CYP1A2         False
 any aromatic N       4040      0.714412 0.803116 0.736965 0.693905 0.697216        CYP2C9        CYP1A2         False

0 of 6 of Beck's isoform assignments appear as the top-enriched isoform in this library.

Every amine class is most enriched in CYP2D6; every aromatic-N class in CYP1

**0 of 6** of Beck's assignments appear as the top-enriched isoform here. Read carefully, that
result says this library's composition offers **no basis for choosing isoform-specific patterns** —
every amine class concentrates in CYP2D6 and every aromatic-N class in CYP1A2, which is a single
axis rather than four isoform-specific ones. It does **not** say Beck are wrong: their claim is
about which moiety coordinates in the molecules that do inhibit, which this measurement cannot
reach. Their own abstract already says such descriptors "alone are insufficient".

Primary amines appear in 39 of 5,655 compounds (0.7%), so the primary-amine arm of their scheme
fails on coverage before any mechanistic question arises.

#### C.6 — the no-match group, and a finding about the whole arm

The compounds with **no** match are the largest group under C1 and the one that decides the
null-handling question. Two things are measured: whether they are a chemically coherent set, and
whether the mere fact of matching carries signal.

In [19]:
# ---- C.6 the no-match group.
nomatch = []
for k in CANDIDATES:
    n = AUD2[f"n_{k}"]
    r = {"candidate": k, "no_match_all": int((n == 0).sum()),
         "frac_all": float((n == 0).mean()),
         "no_match_blind": int((n[AUD2.set == "blind"] == 0).sum()),
         "frac_blind": float((n[AUD2.set == "blind"] == 0).mean())}
    for iso in ISOFORMS:
        sub = AUD2[AUD2[f"has_{iso}"]]
        r[f"frac_{iso}"] = float((sub[f"n_{k}"] == 0).mean())
    nomatch.append(r)
NM = pd.DataFrame(nomatch)
print(NM.to_string(index=False))
NM.to_csv(OUT / "sectionC_no_match.csv", index=False)

# is the no-match group chemically distinctive, or just "everything else"?
no = AUD2[AUD2.n_C1 == 0]
yes = AUD2[AUD2.n_C1 > 0]
props = ["n_nitrogen", "ring_count", "aromatic_ring_count", "heavy_atoms", "mol_wt",
         "rotatable_bonds", "n_basic_n"]
comp = pd.DataFrame([{"property": p, "no_match_mean": no[p].mean(),
                      "matched_mean": yes[p].mean(),
                      "ratio": no[p].mean() / yes[p].mean() if yes[p].mean() else float("nan")}
                     for p in props])
print("\nis the C1 no-match group chemically distinctive?")
print(comp.to_string(index=False))
print(f"\n  no-match compounds containing ANY nitrogen: {int((no.n_nitrogen > 0).sum())}/{len(no)} "
      f"({(no.n_nitrogen > 0).mean():.1%})")
print("  -> 'no match' does not mean 'no nitrogen'. It means no ALIPHATIC BASIC nitrogen, which")
print("     is a narrow structural statement, and the group is otherwise ordinary.")
comp.to_csv(OUT / "sectionC_no_match_properties.csv", index=False)

candidate  no_match_all  frac_all  no_match_blind  frac_blind  frac_CYP1A2  frac_CYP2C9  frac_CYP2D6  frac_CYP3A4
       C1          4642  0.820866             663    0.884000     0.869688     0.873152     0.626256     0.862099
       C2          1189  0.210256             173    0.230667     0.148725     0.217121     0.135968     0.250535
       C3           872  0.154200             137    0.182667     0.114731     0.151751     0.101139     0.176017

is the C1 no-match group chemically distinctive?
           property  no_match_mean  matched_mean    ratio
         n_nitrogen       3.520034      3.427443 1.027015
         ring_count       3.338863      3.263574 1.023070
aromatic_ring_count       2.407152      1.944719 1.237789
        heavy_atoms      24.557087     24.330701 1.009305
             mol_wt     345.894636    343.016722 1.008390
    rotatable_bonds       4.177941      4.739388 0.881536
          n_basic_n       0.000000      1.066140 0.000000

  no-match compounds containi

In [20]:
# ---- C.6b does the presence FLAG alone carry signal? This bears on whether the arm is worth
# building at all, not merely on how to fill a column.
flag_rows = []
tr_lab = train.merge(AUD2[[f"n_{k}" for k in CANDIDATES]], left_on="Molecule_Name",
                     right_index=True)
for k in CANDIDATES:
    for iso in ISOFORMS:
        s = tr_lab[[PIC50_COL[iso], f"n_{k}"]].dropna()
        g1 = s.loc[s[f"n_{k}"] > 0, PIC50_COL[iso]]
        g0 = s.loc[s[f"n_{k}"] == 0, PIC50_COL[iso]]
        if len(g0) < 20 or len(g1) < 20:
            continue
        u, pv = mannwhitneyu(g1, g0)
        flag_rows.append({"candidate": k, "isoform": iso, "n_match": len(g1), "n_no": len(g0),
                          "median_match": g1.median(), "median_no": g0.median(),
                          "rank_biserial": 2 * u / (len(g0) * len(g1)) - 1, "p": pv})
FL = pd.DataFrame(flag_rows)


def bh(p):
    """Benjamini-Hochberg, written out rather than chained -- the one-liner version of this was
    an argsort-of-an-argsort over a re-sorted series and far too easy to get silently wrong."""
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m, dtype=float)
    running = 1.0
    for rank in range(m - 1, -1, -1):
        i = order[rank]
        running = min(running, p[i] * m / (rank + 1))
        adj[i] = running
    return np.clip(adj, 0.0, 1.0)


FL["p_bh"] = FL.groupby("candidate").p.transform(bh)
print(FL.to_string(index=False, float_format="%.4g"))
FL.to_csv(OUT / "sectionC_presence_flag_signal.csv", index=False)
print("\nC1's presence flag alone, per isoform:")
for r in FL[FL.candidate == "C1"].itertuples():
    print(f"  {r.isoform}  rank-biserial {r.rank_biserial:+.3f}  p_BH {r.p_bh:.2e}  "
          f"median {r.median_match:.3f} (match) vs {r.median_no:.3f} (no match)")
print("\nThe sign pattern -- CYP2D6 positive, the other three negative -- independently")
print("reproduces notebook 17's own charge-vs-pIC50 pattern (+0.169 on CYP2D6, negative")
print("elsewhere) from a completely different instrument: one bit of substructure presence.")

candidate isoform  n_match  n_no  median_match  median_no  rank_biserial         p      p_bh
       C1  CYP1A2      184  1228         4.997      5.161        -0.1687   0.00022   0.00022
       C1  CYP2C9      163  1122         4.293      4.669        -0.3272 1.392e-11 2.785e-11
       C1  CYP2D6      558   935         5.036      4.617          0.355 1.494e-30 5.976e-30
       C1  CYP3A4      322  2013         3.891      4.326        -0.1967 1.387e-08 1.849e-08
       C2  CYP1A2     1202   210         5.152      4.974         0.1453 0.0007684  0.001537
       C2  CYP2C9     1006   279         4.608      4.675       -0.03706     0.343    0.4573
       C2  CYP2D6     1290   203         4.785       4.53         0.2886  3.63e-11 1.452e-10
       C2  CYP3A4     1750   585         4.253      4.326       -0.01258    0.6482    0.6482
       C3  CYP1A2     1250   162         5.151      4.974         0.1429  0.003043  0.004057
       C3  CYP2C9     1090   195         4.601      4.746        -0.10

Two findings here, and the second is the most consequential thing in the notebook.

**The no-match group is not chemically distinctive.** 98.5% of them contain nitrogen — just not an
aliphatic basic one — and they are indistinguishable from matched compounds on size, rings,
molecular weight and rotatable bonds. So imputing a value for them cannot be justified by "they
are a different kind of molecule"; they are ordinary molecules lacking one specific motif.

**C1's presence flag alone carries significant signal on all four isoforms**, with CYP2D6
positive (rank-biserial +0.355) and the other three negative, every p_BH ≤ 2.2e-04. That sign
pattern independently reproduces notebook 17's charge-vs-pIC50 result from a completely different
instrument — one bit of substructure presence rather than a pKa model. The implication cuts
against the arm: **much of what a charge-at-the-coordinating-atom column could contribute may
already be carried by a single binary feature**, which is evidence about whether to build the arm
at all, not merely about how to fill its nulls.

The same table carries a second, independent line of evidence on decision 1: **C2's flag is not
significant at all on CYP2C9 (p_BH 0.46) or CYP3A4 (p_BH 0.65)**, where C1's is significant on
every isoform. A pattern matching 79% of compounds does not merely dilute the enrichment measured
in C.1 — it stops separating the labels too.

---

## Part D — looking at the molecules

A histogram cannot show whether a pattern is selecting the right atom. These panels can.

**On the tooling, which is the opposite of what the brief expected.** `mols2grid` 2.2.0 *is*
installed and is this repo's only grid precedent (notebook 02, two call sites) — but it **cannot
highlight a chosen atom per molecule**. Its highlighting is driven by an interactive SMARTS box in
the browser, `prerender=True` is explicitly incompatible with it, and a supplied image column is
ignored. Tested directly: a custom highlight colour does not appear in the rendered output. So
`Draw.MolsToGridImage(highlightAtomLists=...)` is the **primary** tool here despite having no
precedent in this repo, and mols2grid is offered alongside for browsing, with its caption stating
plainly that it does not mark the atom.

**One deviation forced by measurement:** notebook 02 passes `canonical_smiles`. Every panel here
uses the raw `SMILES` column, because Part A.4 measured that the matched atom index differs
between raw and canonical on 8.8% of matched compounds — drawing from canonical would mark the
wrong atom roughly one time in eleven.

In [21]:
# ---- D.0 the panel builder. Highlights the TIE-BREAK WINNER in one colour and the other
# competing atoms in another, so the tie-break can be judged and not just the match.
WIN_C = (0.86, 0.25, 0.24)      # ACCENT-ish: the selected atom
ALT_C = (0.55, 0.70, 0.85)      # the atoms it beat


def panel(names, cand, path, title, rule="hirshfeld", per_row=4, size=(320, 250)):
    """Grid of molecules with the selected atom marked. Raw SMILES throughout -- see Part A.4."""
    ms, legends, hl, hlcol = [], [], [], []
    for n in names:
        m = Chem.MolFromSmiles(smi_lookup[n])
        Chem.rdDepictor.Compute2DCoords(m)
        idxs = list(HITS[cand][n])
        win = None
        if idxs:
            sub = AX.loc[n].reindex(idxs)
            if not sub.q_hirshfeld.isna().any():
                win = int(sub.q_hirshfeld.idxmin() if rule == "hirshfeld"
                          else sub.mulliken_p.idxmax())
        ms.append(m)
        hl.append(idxs)
        hlcol.append({i: (WIN_C if i == win else ALT_C) for i in idxs})
        q = f"  q={AX.loc[(n, win), 'q_hirshfeld']:+.3f}" if win is not None else ""
        legends.append(f"{n}  {len(idxs)} match{'es' if len(idxs) != 1 else ''}{q}")
    # Draw.MolsToGridImage(useSVG=True) SILENTLY DROPS THE LEGENDS in rdkit 2026.03.3 -- the
    # highlights render but the compound name and charge do not, which was caught by rendering a
    # panel and looking at it rather than by trusting the call. The lower-level grid API renders
    # both, so it is used instead and the panels are PNG, matching every other figure here.
    rows_n = (len(ms) + per_row - 1) // per_row
    d = rdMolDraw2D.MolDraw2DCairo(size[0] * per_row, size[1] * rows_n, size[0], size[1])
    o = d.drawOptions()
    # Geometry chosen by rendering the alternatives and looking at them: the default legend
    # fraction leaves the molecule small in a lot of white space and the highlighted atom too
    # small to read at grid scale.
    o.legendFraction = 0.12
    o.legendFontSize = 13
    o.padding = 0.03
    d.DrawMolecules(ms, legends=legends, highlightAtoms=hl, highlightAtomColors=hlcol)
    d.FinishDrawing()
    path.write_bytes(d.GetDrawingText())
    print(f"wrote {path.relative_to(REPO_ROOT)}  ({len(ms)} molecules)")
    return path


def spanning(names, k=8):
    """k compounds spanning the heavy-atom range, deterministically."""
    # sorted() first, so ties in (heavy_atoms, mol_wt) resolve by name and the panels are
    # reproducible across runs rather than depending on set iteration order.
    s = AUD2.loc[sorted(names)].sort_values(["heavy_atoms", "mol_wt", "inchikey"])
    if len(s) <= k:
        return list(s.index)
    return list(s.index[np.linspace(0, len(s) - 1, k).round().astype(int)])


PANEL_DIR = FIG_OUT / "panels"
PANEL_DIR.mkdir(parents=True, exist_ok=True)
print("panel builder ready; selected atom in red, atoms it beat in blue.")

panel builder ready; selected atom in red, atoms it beat in blue.


In [22]:
# ---- D.1 a size-spanning sample for each candidate.
for k in CANDIDATES:
    names = spanning(sets[k], 8)
    panel(names, k, PANEL_DIR / f"D1_spanning_{k}.png", f"{k} size-spanning sample")
display_names = spanning(sets["C1"], 8)
print("\nC1 size-spanning sample:",
      ", ".join(f"{n} ({int(AUD2.loc[n, 'heavy_atoms'])} HA)" for n in display_names))

wrote outputs/43_coordinating_atom/figures/panels/D1_spanning_C1.png  (8 molecules)
wrote outputs/43_coordinating_atom/figures/panels/D1_spanning_C2.png  (8 molecules)
wrote outputs/43_coordinating_atom/figures/panels/D1_spanning_C3.png  (8 molecules)

C1 size-spanning sample: OCNT-2328562 (5 HA), OCNT-0022294 (21 HA), OCNT-2316110 (23 HA), OCNT-2309198 (24 HA), OCNT-2310653 (25 HA), OCNT-2315099 (26 HA), OCNT-2315082 (28 HA), OCNT-2328919 (50 HA)


In [23]:
# ---- D.2 compounds where the candidates disagree about which atom to pick.
# C1 and C2 both match, but select different atom sets -- the widening changed the answer.
disagree = [n for n in (sets["C1"] & sets["C2"])
            if set(HITS["C1"][n]) != set(HITS["C2"][n])]
print(f"{len(disagree)} compounds match both C1 and C2 but select different atom sets "
      f"({len(disagree) / len(sets['C1'] & sets['C2']):.1%} of those matching both)")
sel = spanning(disagree, 8)
panel(sel, "C1", PANEL_DIR / "D2_disagree_asC1.png", "C1's choice")
panel(sel, "C2", PANEL_DIR / "D2_disagree_asC2.png", "C2's choice, same compounds")
pd.DataFrame([{"Molecule_Name": n, "C1_atoms": str(HITS["C1"][n]),
               "C2_atoms": str(HITS["C2"][n])} for n in disagree]).to_csv(
    OUT / "sectionD_candidate_disagreements.csv", index=False)

587 compounds match both C1 and C2 but select different atom sets (57.9% of those matching both)
wrote outputs/43_coordinating_atom/figures/panels/D2_disagree_asC1.png  (8 molecules)
wrote outputs/43_coordinating_atom/figures/panels/D2_disagree_asC2.png  (8 molecules)


In [24]:
# ---- D.3 multi-match compounds, with the two tie-break rules shown side by side. This is the
# panel that lets the tie-break be judged rather than inferred from the 45% disagreement rate.
multi = TB[(TB.candidate == "C2") & (~TB.charge_vs_p)].Molecule_Name.tolist()
print(f"{len(multi)} C2 compounds where most-negative-charge and highest-p pick DIFFERENT atoms")
sel = spanning([n for n in multi], 8)
panel(sel, "C2", PANEL_DIR / "D3_tiebreak_charge.png", "tie-break: most negative charge",
      rule="hirshfeld")
panel(sel, "C2", PANEL_DIR / "D3_tiebreak_ppop.png", "tie-break: highest p population",
      rule="p_pop")

1186 C2 compounds where most-negative-charge and highest-p pick DIFFERENT atoms


wrote outputs/43_coordinating_atom/figures/panels/D3_tiebreak_charge.png  (8 molecules)


wrote outputs/43_coordinating_atom/figures/panels/D3_tiebreak_ppop.png  (8 molecules)


PosixPath('/Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/43_coordinating_atom/figures/panels/D3_tiebreak_ppop.png')

In [25]:
# ---- D.4 the no-match group -- the largest group under C1, and the one that decides null handling.
no_names = [n for n in AUD2.index if not HITS["C1"][n]]
print(f"{len(no_names)} compounds have no C1 match ({len(no_names) / len(AUD2):.1%})")
# show the ones that DO carry nitrogen, since that is the surprise: 98.5% of them do
with_n = [n for n in no_names if AUD2.loc[n, "n_nitrogen"] > 0]
print(f"  of which {len(with_n)} ({len(with_n) / len(no_names):.1%}) contain nitrogen anyway")
panel(spanning(with_n, 8), "C1", PANEL_DIR / "D4_nomatch_but_has_N.png", "no C1 match, has N")
panel(spanning([n for n in no_names if AUD2.loc[n, "n_nitrogen"] == 0], 8), "C1",
      PANEL_DIR / "D4_nomatch_no_N.png", "no C1 match, no N at all")
# and what C2 picks in those same compounds, which is the argument for widening
c2_rescued = [n for n in with_n if HITS["C2"][n]]
print(f"  C2 finds an atom in {len(c2_rescued)} of those {len(with_n)} "
      f"({len(c2_rescued) / len(with_n):.1%}) -- almost always an aromatic N")
panel(spanning(c2_rescued, 8), "C2", PANEL_DIR / "D4_c2_rescues.png", "what C2 picks instead")

4642 compounds have no C1 match (82.1%)
  of which 4573 (98.5%) contain nitrogen anyway


wrote outputs/43_coordinating_atom/figures/panels/D4_nomatch_but_has_N.png  (8 molecules)


wrote outputs/43_coordinating_atom/figures/panels/D4_nomatch_no_N.png  (8 molecules)
  C2 finds an atom in 3453 of those 4573 (75.5%) -- almost always an aromatic N
wrote outputs/43_coordinating_atom/figures/panels/D4_c2_rescues.png  (8 molecules)


PosixPath('/Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/43_coordinating_atom/figures/panels/D4_c2_rescues.png')

In [26]:
# ---- D.5 the interactive grid, following notebook 02's call pattern. It does NOT mark the atom.
import mols2grid
grid_df = pd.DataFrame([
    {"SMILES": smi_lookup[n], "Molecule_Name": n,
     "heavy_atoms": int(AUD2.loc[n, "heavy_atoms"]),
     "n_C1": int(AUD2.loc[n, "n_C1"]), "n_C2": int(AUD2.loc[n, "n_C2"]),
     "n_C3": int(AUD2.loc[n, "n_C3"])}
    for n in spanning(sets["C2"], 24)])
print(f'{len(grid_df)} molecules spanning the C2-matched size range')
print("NOTE: this widget does NOT mark the selected atom -- mols2grid cannot. Use the SVG panels")
print("      above for that. It is here for browsing and for typing a SMARTS into its own box.")
mols2grid.display(grid_df, smiles_col="SMILES",
                  subset=["img", "Molecule_Name", "heavy_atoms", "n_C2"],
                  tooltip=["n_C1", "n_C2", "n_C3"], n_cols=4)

24 molecules spanning the C2-matched size range
NOTE: this widget does NOT mark the selected atom -- mols2grid cannot. Use the SVG panels
      above for that. It is here for browsing and for typing a SMARTS into its own box.


**Panels — `figures/panels/D*.png`.** The selected atom is red; atoms it beat are blue; each legend
carries the match count and the selected atom's Hirshfeld charge.

*D1* shows each candidate across the size range. *D2* shows the same compounds under C1 and under
C2 — the widening does not merely add compounds, it moves the selection in compounds that already
matched. *D3* is the panel that matters most for decision 2: the same compounds under each
tie-break rule, restricted to the cases where the rules disagree, so the choice can be judged
chemically rather than from the 45% disagreement rate alone. *D4* shows the no-match group, split
by whether the compound contains nitrogen at all — and almost all of them do, which is why
"no match" is a statement about one motif rather than about the molecule.

The `mols2grid` widget above is for browsing only and deliberately does not claim to show the
selection.

---

## Part E — the three decisions, the evidence, and no choice made

Each decision below is stated with the numbers that bear on it. **None is resolved here.** The
first thing to say is that they are not three independent choices: decisions 1 and 3 are two ends
of one trade-off, and the weight of decision 2 depends entirely on how decision 1 goes.

In [27]:
# ---- E.1 assemble the evidence for each decision from the saved tables, not from prose, so the
# verdict cannot drift from the sections.
V = {}

cov = pd.read_csv(OUT / "sectionC_coverage.csv").set_index("candidate")
dist = pd.read_csv(OUT / "sectionC_match_distribution.csv").set_index("candidate")
nm = pd.read_csv(OUT / "sectionC_no_match.csv").set_index("candidate")
tbs = pd.read_csv(OUT / "sectionC_tiebreak_agreement.csv").set_index("candidate")
fl = pd.read_csv(OUT / "sectionC_presence_flag_signal.csv")

print("=" * 94)
print("DECISION 1 -- WHICH PATTERN.  No candidate is both faithful to the chemistry and selective.")
print("=" * 94)
for k, (desc, _) in CANDIDATES.items():
    others = np.mean([cov.loc[k, i] for i in ISOFORMS if i != "CYP2D6"])
    print(f"\n  {k}  {desc}")
    print(f"      coverage {cov.loc[k, 'fraction']:.1%} of all compounds; "
          f"empty for {nm.loc[k, 'frac_all']:.1%} of the library and "
          f"{nm.loc[k, 'frac_blind']:.1%} of the scored blind 750")
    print(f"      CYP2D6 enrichment {cov.loc[k, 'CYP2D6'] / others:.2f}x  "
          f"(per isoform: " + ", ".join(f"{i} {cov.loc[k, i]:.3f}" for i in ISOFORMS) + ")")
V["decision_1_which_pattern"] = {
    "resolved": False,
    "the_bind": ("C1 selects (2.84x CYP2D6 enrichment) but is empty for 82.1% of the library and "
                 "88.4% of the scored blind set. C2 is the faithful reading of Correia & "
                 "Hollenberg -- who put aliphatic and aromatic N on equal footing and specify no "
                 "substitution class -- but at 79% coverage its per-isoform spread is flat, so it "
                 "selects nothing. C3 adds mostly hydroxyl oxygen, the one donor class the "
                 "chapter itself calls 'generally weak'."),
    "per_candidate": {k: {"coverage": float(cov.loc[k, "fraction"]),
                          "empty_fraction_all": float(nm.loc[k, "frac_all"]),
                          "empty_fraction_blind": float(nm.loc[k, "frac_blind"]),
                          "cyp2d6_enrichment": float(
                              cov.loc[k, "CYP2D6"]
                              / np.mean([cov.loc[k, i] for i in ISOFORMS if i != "CYP2D6"])),
                          "per_isoform_coverage": {i: float(cov.loc[k, i]) for i in ISOFORMS}}
                      for k in CANDIDATES}}

DECISION 1 -- WHICH PATTERN.  No candidate is both faithful to the chemistry and selective.

  C1  conservative: aliphatic basic N only
      coverage 17.9% of all compounds; empty for 82.1% of the library and 88.4% of the scored blind 750
      CYP2D6 enrichment 2.84x  (per isoform: CYP1A2 0.130, CYP2C9 0.127, CYP2D6 0.374, CYP3A4 0.138)

  C2  faithful: + aromatic/heteroaromatic N
      coverage 79.0% of all compounds; empty for 21.0% of the library and 23.1% of the scored blind 750
      CYP2D6 enrichment 1.09x  (per isoform: CYP1A2 0.851, CYP2C9 0.783, CYP2D6 0.864, CYP3A4 0.749)

  C3  inclusive: + divalent S and hydroxyl O
      coverage 84.6% of all compounds; empty for 15.4% of the library and 18.3% of the scored blind 750
      CYP2D6 enrichment 1.05x  (per isoform: CYP1A2 0.885, CYP2C9 0.848, CYP2D6 0.899, CYP3A4 0.824)


In [28]:
print("=" * 94)
print("DECISION 2 -- THE TIE-BREAK.  Nearly free under C1; the larger decision under C2 or C3.")
print("=" * 94)
for k in CANDIDATES:
    print(f"\n  {k}  {dist.loc[k, 'two_or_more']:.1%} of compounds need a tie-break "
          f"({int(dist.loc[k, 'n_needing_tiebreak']):,} compounds, up to "
          f"{int(dist.loc[k, 'max'])} competing atoms)")
    print(f"      most-negative-charge vs highest-p disagree on "
          f"{1 - tbs.loc[k, 'agree_charge_vs_p']:.1%} of them")
    print(f"      and the CHARGE SCHEME itself disagrees on "
          f"{1 - tbs.loc[k, 'agree_hirshfeld_vs_mulliken']:.1%} (Hirshfeld vs Mulliken)")
print("\n  -> three axes, not one: the rule, the scheme, and whether a tie-break is needed at all.")
V["decision_2_tiebreak"] = {
    "resolved": False,
    "note": ("The two proposed rules are close to independent of each other where a tie-break is "
             "needed, and the choice of charge SCHEME is a third axis the brief did not "
             "anticipate, disagreeing nearly as often as the choice of rule. Panel D3 shows the "
             "disagreeing cases so the choice can be made chemically."),
    "per_candidate": {k: {"frac_needing_tiebreak": float(dist.loc[k, "two_or_more"]),
                          "n_needing_tiebreak": int(dist.loc[k, "n_needing_tiebreak"]),
                          "max_competing_atoms": int(dist.loc[k, "max"]),
                          "charge_vs_p_disagree": float(1 - tbs.loc[k, "agree_charge_vs_p"]),
                          "hirshfeld_vs_mulliken_disagree": float(
                              1 - tbs.loc[k, "agree_hirshfeld_vs_mulliken"])}
                      for k in CANDIDATES}}

DECISION 2 -- THE TIE-BREAK.  Nearly free under C1; the larger decision under C2 or C3.

  C1  1.2% of compounds need a tie-break (67 compounds, up to 2 competing atoms)
      most-negative-charge vs highest-p disagree on 46.3% of them
      and the CHARGE SCHEME itself disagrees on 47.8% (Hirshfeld vs Mulliken)

  C2  44.0% of compounds need a tie-break (2,491 compounds, up to 6 competing atoms)
      most-negative-charge vs highest-p disagree on 47.6% of them
      and the CHARGE SCHEME itself disagrees on 53.6% (Hirshfeld vs Mulliken)

  C3  51.8% of compounds need a tie-break (2,929 compounds, up to 7 competing atoms)
      most-negative-charge vs highest-p disagree on 43.4% of them
      and the CHARGE SCHEME itself disagrees on 45.1% (Hirshfeld vs Mulliken)

  -> three axes, not one: the rule, the scheme, and whether a tie-break is needed at all.


In [29]:
print("=" * 94)
print("DECISION 3 -- NULL HANDLING.  A modelling decision, and the evidence complicates the arm.")
print("=" * 94)
print(f"\n  Under C1 the column is empty for {nm.loc['C1', 'frac_all']:.1%} of the library and "
      f"{nm.loc['C1', 'frac_blind']:.1%} of the scored blind 750.")
print("  Per isoform: " + ", ".join(
    f"{i} {nm.loc['C1', f'frac_{i}']:.1%}" for i in ISOFORMS))
print(f"\n  The no-match group is NOT chemically distinctive: "
      f"{(no.n_nitrogen > 0).mean():.1%} of them contain nitrogen, and they match the matched")
print("  group on size, rings, molecular weight and rotatable bonds. So imputation cannot be")
print("  justified by 'they are a different kind of molecule'.")
print("\n  But the PRESENCE FLAG alone is significant on all four isoforms:")
for r in fl[fl.candidate == "C1"].itertuples():
    print(f"     {r.isoform}  rank-biserial {r.rank_biserial:+.3f}  p_BH {r.p_bh:.2e}")
print("\n  -> that sign pattern (CYP2D6 positive, the rest negative) independently reproduces")
print("     notebook 17's charge-vs-pIC50 result from a different instrument. Much of what the")
print("     value column could contribute may already be in one bit, which is evidence about")
print("     whether to build the arm at all -- not just about how to fill it.")
V["decision_3_null_handling"] = {
    "resolved": False,
    "empty_fraction_all_C1": float(nm.loc["C1", "frac_all"]),
    "empty_fraction_blind_C1": float(nm.loc["C1", "frac_blind"]),
    "empty_fraction_per_isoform_C1": {i: float(nm.loc["C1", f"frac_{i}"]) for i in ISOFORMS},
    "no_match_group_contains_nitrogen": float((no.n_nitrogen > 0).mean()),
    "no_match_group_is_chemically_distinctive": False,
    "presence_flag_signal_C1": {
        r.isoform: {"rank_biserial": float(r.rank_biserial), "p_bh": float(r.p_bh)}
        for r in fl[fl.candidate == "C1"].itertuples()},
    "implication": ("The presence flag alone is significant on all four isoforms and reproduces "
                    "notebook 17's sign pattern, so a one-bit feature may already carry much of "
                    "what the charge column would add. This is evidence about whether the arm is "
                    "worth building, not only about null handling.")}

DECISION 3 -- NULL HANDLING.  A modelling decision, and the evidence complicates the arm.

  Under C1 the column is empty for 82.1% of the library and 88.4% of the scored blind 750.
  Per isoform: CYP1A2 87.0%, CYP2C9 87.3%, CYP2D6 62.6%, CYP3A4 86.2%

  The no-match group is NOT chemically distinctive: 98.5% of them contain nitrogen, and they match the matched
  group on size, rings, molecular weight and rotatable bonds. So imputation cannot be
  justified by 'they are a different kind of molecule'.

  But the PRESENCE FLAG alone is significant on all four isoforms:
     CYP1A2  rank-biserial -0.169  p_BH 2.20e-04
     CYP2C9  rank-biserial -0.327  p_BH 2.78e-11
     CYP2D6  rank-biserial +0.355  p_BH 5.98e-30
     CYP3A4  rank-biserial -0.197  p_BH 1.85e-08

  -> that sign pattern (CYP2D6 positive, the rest negative) independently reproduces
     notebook 17's charge-vs-pIC50 result from a different instrument. Much of what the
     value column could contribute may already be in one

In [30]:
# ---- E.2 the verdict file and the one-line-per-section summary.
V["recommendation"] = {
    "is_a_recommendation_not_a_choice": True,
    "text": ("If the arm is built at all, C1 is the only candidate that selects anything, and its "
             "tie-break is nearly free (1.2% of compounds). Its cost is a column empty for 88.4% "
             "of the scored blind set, and the honest reading of Part C.6 is that a presence flag "
             "may capture most of the available signal for one bit instead of a QM pipeline. C2 "
             "is what the cited chemistry actually says and is measurably useless as a feature at "
             "79% coverage with no isoform spread. That tension is the finding; it is not "
             "resolved here."),
    "what_would_settle_it": ("An OOF ST-RAE comparison on this project's frozen folds -- the same "
                             "bar notebook 41 set for CYP1A2/CYP2C9's charge signal, and the bar "
                             "notebook 18's clean tie failed to clear.")}
V["meta"] = {
    "notebook": "43_coordinating_atom_candidates",
    "n_train": int(len(TR)), "n_blind": int(len(BL)), "n_total": int(len(AUD)),
    "rdkit": rdkit.__version__,
    "quotations_verified": f"{n_ok}/{len(QV)}",
    "written_at_utc": datetime.now(timezone.utc).isoformat(),
    "nothing_computed": ["model training", "feature file for modelling", "pattern selection",
                         "submission"],
}
(OUT / "verdict.json").write_text(json.dumps(V, indent=2))
print(f"-> verdict.json ({len(V)} top-level keys)")

SUMMARY = pd.DataFrame([
    {"section": "0", "headline": f"{n_ok}/{len(QV)} quotations verified verbatim; the brief's "
     f"'neutral species' premise corrected", "file": "quote_verification.json"},
    {"section": "A", "headline": f"{len(A):,} atom / {len(BND):,} bond / {len(MOL):,} molecular "
     f"rows recomputed; {n_two:,} Cl/Br atoms across {n_two_cmp} compounds, matching notebook "
     f"41's independent count", "file": "sectionA_trap_acceptance.csv"},
    {"section": "B", "headline": "C1 reproduces notebook 41's strict pattern on 5,655/5,655; the "
     "cited chemistry actually favours C2, not C1", "file": "sectionB_candidates.csv"},
    {"section": "C", "headline": f"C1 {cov.loc['C1', 'fraction']:.1%} coverage with 2.84x CYP2D6 "
     f"enrichment; C2 {cov.loc['C2', 'fraction']:.1%} with none", "file": "sectionC_coverage.csv"},
    {"section": "C.4", "headline": f"the two tie-break rules disagree on "
     f"{1 - tbs.loc['C2', 'agree_charge_vs_p']:.0%} of multi-match compounds",
     "file": "sectionC_tiebreak_agreement.csv"},
    {"section": "C.5", "headline": f"{int(BK.matches_beck.sum())}/6 of Beck's isoform assignments "
     f"appear as top-enriched here (a composition result, not a refutation)",
     "file": "sectionC_beck_isoform_check.csv"},
    {"section": "C.6", "headline": "the presence flag alone is significant on all four isoforms "
     "and reproduces notebook 17's sign pattern",
     "file": "sectionC_presence_flag_signal.csv"},
    {"section": "D", "headline": "mols2grid cannot mark a chosen atom; RDKit grids used as primary",
     "file": "figures/panels/"},
    {"section": "E", "headline": "all three decisions left open, with the evidence assembled",
     "file": "verdict.json"}])
SUMMARY.to_csv(OUT / "audit_summary.csv", index=False)
print(SUMMARY.to_string(index=False))

pd.DataFrame([{"candidate": k, "description": d, "smarts": s}
              for k, (d, s) in CANDIDATES.items()]).to_csv(
    OUT / "sectionB_candidates.csv", index=False)
# the reusable per-compound table: match counts and the selected atom under each rule
PERC = AUD2[["set", "inchikey", "heavy_atoms", "n_nitrogen", "n_basic_n"]
            + [f"n_{k}" for k in CANDIDATES]
            + [f"has_{i}" for i in ISOFORMS]].copy()
for k in CANDIDATES:
    PERC[f"atoms_{k}"] = pd.Series({n: ",".join(map(str, v)) for n, v in HITS[k].items()})
PERC.to_csv(OUT / "per_compound_candidates.csv")
print(f"\n-> per_compound_candidates.csv ({len(PERC)} rows, {PERC.shape[1]} cols)")

-> verdict.json (5 top-level keys)
section                                                                                                                                      headline                              file
      0                                                           18/18 quotations verified verbatim; the brief's 'neutral species' premise corrected           quote_verification.json
      A 252,237 atom / 269,256 bond / 5,655 molecular rows recomputed; 924 Cl/Br atoms across 815 compounds, matching notebook 41's independent count      sectionA_trap_acceptance.csv
      B                                    C1 reproduces notebook 41's strict pattern on 5,655/5,655; the cited chemistry actually favours C2, not C1           sectionB_candidates.csv
      C                                                                            C1 17.9% coverage with 2.84x CYP2D6 enrichment; C2 79.0% with none             sectionC_coverage.csv
    C.4                                      

---

### What this does not establish

- **No pattern is chosen, and no feature file for modelling is written.** The three decisions in
  Part E are stated with their evidence and left open deliberately.
- **A 2D substructure match cannot establish that coordination occurs.** The strongest available
  caveat is the chapter's own: 1,2,3-triazole coordinates the CYP3A4 iron when free but binds
  through a **bridging water** when tethered to ethinylestradiol, giving a metabolically competent
  complex (§5.2.4, verified verbatim in Part 0.2). The same nitrogen, two different outcomes,
  decided by steric context no SMARTS can see.
- **Coverage is not signal.** Notebook 18 gave a model exactly the charge descriptor notebook 17's
  mechanism argued for and got a clean tie on all four isoforms with pool-typical error
  correlation. Adequate coverage has already failed to predict usefulness once in this project.
- **Part C.5 does not refute Beck et al.** It measures this library's composition, which is a
  property of how the compound sets were assembled. Their claim is about which moiety coordinates
  in molecules that do inhibit, and nothing here reaches it.
- **The quotations are verified but not independently reproducible.** Neither source is in the
  repository; both sha256 values pin what was read on this machine only.
- **The presence-flag result is univariate and on training labels.** It shows a flag correlates
  with pIC50; it does not show a model would use it, nor that it would beat the value column.
- **Nothing here measures blind performance.** No OOF comparison was run; that is the bar that
  would actually settle decision 1 and it is out of scope.

### Decisions taken without approval

Everything the brief did not pin down, with the reason and the alternative that was not taken.

In [31]:
record_decision(
    decision="Reframed the candidate ladder: C2, not C1, is the faithful reading of the chemistry.",
    reason=("The brief presented the conservative end as 'aliphatic amine only', implying that is "
            "what the sources support. Verified verbatim, Correia & Hollenberg name the donor set "
            "as 'Amino-side chains or nitrogenous heterocyclic moieties' (p.178) and restate it as "
            "'nitrogen-containing aliphatic or aromatic functions that bind the heme iron tightly' "
            "(p.182) -- aliphatic and aromatic on equal footing, and no substitution class "
            "anywhere. Presenting C1 as the chemistry-grounded option would have inverted the "
            "actual relationship between the three candidates, which is the substance of "
            "decision 1."),
    alternatives=("Present C1 as grounded and C2/C3 as speculative widenings (would have hidden "
                  "that the selective candidate is the unprincipled one, and made the central "
                  "bind invisible)."),
    authority="the brief's own instruction to read both sources before writing the patterns")

record_decision(
    decision=("Used Draw.MolsToGridImage as the primary renderer for Part D, with mols2grid "
              "alongside for browsing only."),
    reason=("The brief expected mols2grid to be absent and RDKit grids to be the fallback. "
            "Measured: mols2grid 2.2.0 IS installed and is this repo's only grid precedent "
            "(notebook 02), but it cannot highlight a chosen atom per molecule -- highlighting is "
            "driven by an interactive SMARTS box, prerender=True is explicitly incompatible with "
            "it, and a supplied image column is ignored (a custom highlight colour does not "
            "appear in the output). Showing WHICH atom was selected is Part D's entire purpose, "
            "so the brief's fallback had to become the primary tool despite having no precedent "
            "here."),
    alternatives=("mols2grid alone, following the repo precedent (would have shown which "
                  "compounds matched but never which atom, so decision 2 could not be judged "
                  "visually at all); RDKit grids alone (loses interactive browsing)."),
    authority="user-confirmed before building, after the limitation was measured")

record_decision(
    decision="Read the raw SMILES column everywhere, deviating from notebook 02's grid pattern.",
    reason=(f"Notebook 02 passes canonical_smiles to mols2grid. Measured over all {N_POP:,} "
            f"compounds: the raw column reproduces ORCA's atom order {ok_raw:,}/{N_POP:,} and "
            f"canonical_smiles only {ok_can:,}/{N_POP:,}, failing on {N_POP - ok_can:,} "
            f"({pct(N_POP - ok_can, N_POP):.1f}%). Among C1-matched compounds the matched atom "
            f"INDEX differs between the two on {idx_diff} of {n_matched} ({pct(idx_diff, n_matched):.1f}%). "
            f"Since every tie-break and every panel joins a SMARTS atom index into qm_atoms.csv, "
            f"canonical_smiles would have returned the charge on the wrong atom about one time in "
            f"eleven -- silently, with no error raised."),
    alternatives=("Follow notebook 02 and use canonical_smiles (silently wrong on ~9% of matched "
                  "compounds, with no error raised)."),
    authority="measured; the hazard is specific to atom-index-dependent work")

record_decision(
    decision="Added the presence-flag signal test (C.6b), which the brief did not ask for.",
    reason=("The brief frames null handling as 'zero, imputation, or a presence flag plus the "
            "value'. Measuring whether the flag ALONE carries signal turns that from a "
            "data-hygiene question into evidence about the arm itself -- and it does, on all four "
            "isoforms, reproducing notebook 17's own sign pattern from a different instrument. "
            "Leaving it out would have presented decision 3 as narrower than it is."),
    alternatives=("Report only the null fractions as briefed (would have omitted the strongest "
                  "evidence bearing on whether to build the arm at all)."),
    authority="none -- added because it materially changes decision 3")

record_decision(
    decision="Verified book quotations against the union of three pdftotext geometries.",
    reason=("The book is a two-column scan. Tested: a full-page extract interleaves the columns "
            "and corrupts sentences spanning a column break, while a hard x=240 split clips text "
            "on the boundary. No single geometry yields all 13 quotations. The p.179 lipophilicity "
            "sentence is split across a column break with figure text interleaved and cannot be "
            "recovered contiguously at all, so it is verified as two fragments and labelled as "
            "such."),
    alternatives=("Use one geometry and silently drop the quotes it misses; or paraphrase rather "
                  "than quote (which is how the premise error in Part 0.3 arose in the first "
                  "place)."),
    authority="measured; notebook 37's quote-verification convention adapted to a scanned source")

print(f"{len(DECISIONS)} decisions recorded in decisions_taken.json\n")
for i, d in enumerate(DECISIONS, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

6 decisions recorded in decisions_taken.json

1. Corrected the brief's claim that Correia & Hollenberg support computing on the neutral species, and re-attributed that choice to this project rather than to the source.
   WHY: The brief read 'the neutral ferric P450 species' as licensing the neutral LIGAND microspecies. Verified verbatim against the PDF (sha256 e69527f273955630...), 'neutral' qualifies the enzyme's redox state -- ferric vs ferrous -- and the passage concerns anions (cyanide). Beck et al. say the opposite for CYP2D6: inhibitors 'capable of procuring a positive charge', with 'a predilection for basic substrates'. Leaving the attribution as briefed would have put a citation behind a claim neither source makes, in a notebook whose whole purpose is to ground a pattern in cited chemistry. This is the fifth brief-premise error caught in this project, after notebooks 30, 32, 36 and 38.
   ALTERNATIVES: Repeat the brief's attribution (a false citation in the one place it would m

### Scope check — fails loudly

Four tests, following notebooks `30`/`34`/`35`/`37`/`38`/`39`/`41`/`42`'s fixed version. **TEST 1**
looks only at *tracked* content changes under a protected path, minus those already dirty before
this run, so neither a pre-existing untracked directory nor a pre-existing modification can raise
a false positive. **TEST 2** checks nothing inside a protected tree was written during this run,
against the timestamp recorded at the top rather than one inferred from an mtime at the bottom.
**TEST 3** confirms both notebooks carrying live submission code were never modified. **TEST 4**
confirms no submission-shaped file and no modelling feature file was created.

In [32]:
st_out = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                        cwd=REPO_ROOT).stdout
print("git status --porcelain:")
print(st_out if st_out.strip() else "(clean)")

PROTECTED = [
    "data/", "src/", "scripts/",
    "docs/",
    "outputs/qm_parsed/", "outputs/qm_descriptors/",
    "notebooks/30_aid_full_retrain.ipynb", "outputs/30_aid_full_retrain/",
    "notebooks/32_deadzone_aid_retrain.ipynb", "outputs/32_deadzone_aid_retrain/",
    "notebooks/33_butina_cluster_split.ipynb", "outputs/33_butina_split/",
    "notebooks/34_butina_5x5_repeated_cv.ipynb", "outputs/34_butina_5x5/",
    "notebooks/35_butina_repeat3.ipynb", "outputs/35_butina_repeat3/",
    "notebooks/36_spread_corrected_nb32a.ipynb", "outputs/36_spread_corrected_nb32a/",
    "notebooks/37_butina_5x5_complete.ipynb", "outputs/37_butina_5x5_complete/",
    "notebooks/38_cyp3a4_placement_diagnostic.ipynb", "outputs/38_cyp3a4_placement_diagnostic/",
    "notebooks/39_ambiguity_and_shared_blend.ipynb", "outputs/39_ambiguity_and_shared_blend/",
    "notebooks/40_single_object_ensemble.ipynb", "outputs/40_single_object_ensemble/",
    "notebooks/41_dataset_audit_3d.ipynb", "outputs/41_dataset_audit_3d/",
    "notebooks/42_orca_timing_run.ipynb", "outputs/42_orca_timing/",
    "outputs/17_protonation_charge_check/",
    "outputs/submissions/", "outputs/board_solved_calibration/",
]

tracked_violations = [l for l in st_out.splitlines()
                      if any(x in l[3:].strip() for x in PROTECTED)
                      and l[:2].strip() != "??"
                      and l[3:].strip() not in DIRTY_AT_START]
written = []
for prot in PROTECTED:
    q = REPO_ROOT / prot
    if not q.exists():
        continue
    for fp in ([q] if q.is_file() else [x for x in q.rglob("*") if x.is_file()]):
        try:
            if fp.stat().st_mtime > RUN_STARTED_AT:
                written.append(str(fp.relative_to(REPO_ROOT)))
        except OSError:
            pass

print(f"\nTEST 1 -- tracked modifications under a protected path, EXCLUDING the "
      f"{len(DIRTY_AT_START)} already dirty before this run: {len(tracked_violations)}")
for t in tracked_violations:
    print(f"    VIOLATION: {t}")
print(f"TEST 2 -- files written inside a protected tree during this run: {len(written)}")
for t in written[:25]:
    print(f"    VIOLATION: {t}")

LIVE_SUBMISSION_NOTEBOOKS = ["notebooks/32_deadzone_aid_retrain.ipynb",
                             "notebooks/40_single_object_ensemble.ipynb"]
live_ok = True
for rel in LIVE_SUBMISSION_NOTEBOOKS:
    p = REPO_ROOT / rel
    ok_ = (not p.exists()) or p.stat().st_mtime <= RUN_STARTED_AT
    live_ok &= ok_
    print(f"TEST 3 -- {rel} unmodified: {ok_}")

made = sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file())
bad = [m for m in made if "submission" in m.lower()
       or m.endswith((".npy", ".parquet")) and "feature" in m.lower()]
print(f"TEST 4 -- submission- or feature-file-shaped outputs created: {len(bad)} {bad}")

if tracked_violations or written or not live_ok or bad:
    raise ValueError("scope violated -- investigate before trusting anything above")
print("\nconfirmed: no protected path modified, both live-submission notebooks untouched, "
      "no submission or modelling feature file created.")
print(f"\noutputs written ({len(made)} files under {OUT.relative_to(REPO_ROOT)}/):")
for m in made:
    print(f"  {m}")

git status --porcelain:
 M .gitignore
?? notebooks/43_coordinating_atom_candidates.ipynb
?? outputs/43_coordinating_atom/
?? outputs/qm_parsed/
?? scripts/parse_orca_output.py




TEST 1 -- tracked modifications under a protected path, EXCLUDING the 1 already dirty before this run: 0
TEST 2 -- files written inside a protected tree during this run: 0
TEST 3 -- notebooks/32_deadzone_aid_retrain.ipynb unmodified: True
TEST 3 -- notebooks/40_single_object_ensemble.ipynb unmodified: True
TEST 4 -- submission- or feature-file-shaped outputs created: 0 []

confirmed: no protected path modified, both live-submission notebooks untouched, no submission or modelling feature file created.

outputs written (41 files under outputs/43_coordinating_atom/):
  audit_summary.csv
  decisions_taken.json
  figures/atom_count_reproduction.png
  figures/charge_distributions_by_scheme.png
  figures/mayer_bond_order_split.png
  figures/panels/D1_spanning_C1.png
  figures/panels/D1_spanning_C2.png
  figures/panels/D1_spanning_C3.png
  figures/panels/D2_disagree_asC1.png
  figures/panels/D2_disagree_asC2.png
  figures/panels/D3_tiebreak_charge.png
  figures/panels/D3_tiebreak_ppop.png
  f

In [33]:
print("=" * 84)
print("NOTEBOOK 43 COMPLETE -- candidate definitions and evidence only")
print("=" * 84)
print(f"""
Part A   the ORCA parse on record: {len(A):,} atom rows, {len(BND):,} bond rows, {len(MOL):,}
         molecular rows, all recomputed here. {n_two:,} Cl/Br atoms across {n_two_cmp} compounds,
         matching notebook 41's independently derived count -- the fixed-column reader's
         acceptance test. 12 cross-checks against the generating run, 9 at exactly 0.000e+00.

Part 0.3 the brief's claim that Correia & Hollenberg support computing on the neutral LIGAND
         species is wrong: 'neutral' qualifies the ferric ENZYME, and Beck et al. say CYP2D6
         inhibitors carry a positive charge. The QM run's choice stands; its attribution does not.

Part B   C1 reproduces notebook 41's strict pattern on 5,655/5,655. The cited chemistry puts
         aliphatic and aromatic N on equal footing, so C2 -- not C1 -- is its faithful reading.

Part C   C1 {cov.loc['C1', 'fraction']:.1%} coverage, 2.84x CYP2D6 enrichment, empty for
         {nm.loc['C1', 'frac_blind']:.1%} of the scored blind 750.
         C2 {cov.loc['C2', 'fraction']:.1%} coverage, no isoform spread at all.
         The two tie-break rules disagree on {1 - tbs.loc['C2', 'agree_charge_vs_p']:.0%} of
         multi-match compounds; the charge scheme disagrees nearly as often.
         The presence flag ALONE is significant on all four isoforms.

Part D   {len(list((FIG_OUT / 'panels').glob('*.png')))} panels with the selected atom marked.
         mols2grid cannot mark a chosen atom -- measured, not assumed.

Part E   all three decisions left OPEN, with the evidence assembled in verdict.json.

NOTHING trained. NO feature file for modelling. NO pattern chosen. NO submission built,
validated or sent.
""")

NOTEBOOK 43 COMPLETE -- candidate definitions and evidence only

Part A   the ORCA parse on record: 252,237 atom rows, 269,256 bond rows, 5,655
         molecular rows, all recomputed here. 924 Cl/Br atoms across 815 compounds,
         matching notebook 41's independently derived count -- the fixed-column reader's
         acceptance test. 12 cross-checks against the generating run, 9 at exactly 0.000e+00.

Part 0.3 the brief's claim that Correia & Hollenberg support computing on the neutral LIGAND
         species is wrong: 'neutral' qualifies the ferric ENZYME, and Beck et al. say CYP2D6
         inhibitors carry a positive charge. The QM run's choice stands; its attribution does not.

Part B   C1 reproduces notebook 41's strict pattern on 5,655/5,655. The cited chemistry puts
         aliphatic and aromatic N on equal footing, so C2 -- not C1 -- is its faithful reading.

Part C   C1 17.9% coverage, 2.84x CYP2D6 enrichment, empty for
         88.4% of the scored blind 750.
         